# 第三阶段：冻结 T0 初解后的连续功率优化实际代码

本笔记本给想读懂 **S3-P03 正式实验代码** 的读者：按“固定频率槽 → 物理容量与 SI/SGM → 旧 SCA（A）→ SI 优先追加段（B/C）→ 工作台保存”的顺序阅读。代码单元是冻结提交 `e85a9429461a358ba2e11614a3cce4671d12b68e` 的**完整原始模块**，每格的 `frozen_source` 元数据给出文件、行数及 SHA256。运行时再次逐字节核对 Git blob，任何改动都会报错。核心代码较长，可折叠单元格，并按文中的函数名定位。

**默认 Run All 只导出冻结源码、加载函数/类并核对配置，不调用 T0 模型、优化器或正式评价。** 导出位置固定为 `outputs/training_notebooks_20261012/stage3/frozen_src/`。此处是教学用可执行源码，不是新的训练或实验结果。要重做正式评价，仍须在授权范围内经实验工作台创建任务。阅读前需要本仓库 Git 及 Python；真正求解还需要冻结 DQN/CUDA、SciPy、CVXPY/Clarabel、场景和 T0 权重。

依据：[S3-P03 运行清单](../../../outputs/experiments/stage3_si_priority_20261003/run_manifest.json)、[正式报告03](../实验报告/03_SI优先功率细调与容量近似对照.md)。本轮两预算、共同 Jan27 的 100 景×3 动作×A/B/C 为 1800 条方法结果，**新增训练交互 0**。

## 0. 安全导出冻结源码并绑定导入路径

以下代码用 Git 归档导出**整个**冻结 `src/satellite_rl` 到隔离目录；所有 Python 依赖仍从该版本导入。若当前内核先前已经导入另一版 `satellite_rl`，请重启内核。只创建/核验教学目录，不改工作树源码、模型或正式实验资产。

In [1]:
from pathlib import Path
import hashlib
import io
import json
import subprocess
import sys
import zipfile

SCIENCE_COMMIT = "e85a9429461a358ba2e11614a3cce4671d12b68e"
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path("D:/paper")]
    for candidate in candidates:
        if (candidate / "EXPERIMENT_STATUS.md").is_file() and (candidate / ".git").exists():
            return candidate.resolve()
    raise FileNotFoundError("请在 D:/paper 仓库或其子目录启动 Jupyter")

PROJECT_ROOT = find_project_root()
ISOLATED = PROJECT_ROOT / "outputs/training_notebooks_20261012/stage3/frozen_src"
if any(name == "satellite_rl" or name.startswith("satellite_rl.") for name in sys.modules):
    raise RuntimeError("内核已有 satellite_rl；请重启内核再 Run All")

archive = subprocess.check_output(
    ["git", "archive", "--format=zip", SCIENCE_COMMIT, "src/satellite_rl"],
    cwd=PROJECT_ROOT)
ISOLATED.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as zipped:
    for item in zipped.infolist():
        target = (ISOLATED / item.filename).resolve()
        if not target.is_relative_to(ISOLATED.resolve()):
            raise RuntimeError(f"归档路径越界: {item.filename}")
        if item.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            payload = zipped.read(item)
            if target.exists() and target.read_bytes() != payload:
                raise RuntimeError(f"隔离目录中已有不同内容: {target}")
            target.write_bytes(payload)
sys.path.insert(0, str(ISOLATED / "src"))
print("冻结源码:", SCIENCE_COMMIT)
print("隔离路径:", ISOLATED)


冻结源码: e85a9429461a358ba2e11614a3cce4671d12b68e
隔离路径: D:\paper\outputs\training_notebooks_20261012\stage3\frozen_src


## 1. 冻结配置：A/B/C、预算和安全门槛

`config.py` 真正校验前端传入的参数。S3-P03 的方法顺序为 `sca`（A）、`sca_si_lp`（B）、`sca_si_lb`（C）；A 与 B/C 追加段各最多 2000 次容量调用、300 轮和 30 秒。两种预算分别是整景 3000 W，及冻结 T0 初始总功率。这里没有训练参数更新。下面定义一个教学用 cell magic，把各格**原始完整源码**编译进其原模块命名空间；运行前与 Git blob 原文逐字节比较。

In [2]:
import importlib
from IPython.core.magic import register_cell_magic

@register_cell_magic
def project_module(line, cell):
    module_name = line.strip()
    prefix = "satellite_rl.power_refinement."
    if not module_name.startswith(prefix):
        raise ValueError(module_name)
    path = "src/" + module_name.replace(".", "/") + ".py"
    original = subprocess.check_output(
        ["git", "show", f"{SCIENCE_COMMIT}:{path}"], cwd=PROJECT_ROOT).decode("utf-8")
    if cell != original:
        raise RuntimeError(f"代码格与冻结 Git blob 不一致: {path}")
    module = importlib.import_module(module_name)
    if not Path(module.__file__).resolve().is_relative_to(ISOLATED.resolve()):
        raise RuntimeError(f"模块未从隔离源码加载: {module.__file__}")
    exec(compile(cell, f"{SCIENCE_COMMIT}:{path}", "exec"), module.__dict__)
    print(path, "行数", len(cell.splitlines()), "SHA256", hashlib.sha256(cell.encode()).hexdigest())


## 1.1 实际任务配置及校验

查看 `SI_PRIORITY_METHODS`、`DEFAULTS`、`validate_config`。特别注意 B/C 只能和 A 组成规定的同源方法组。

以下代码格为冻结 `src/satellite_rl/power_refinement/config.py` 的全部源码，保留原函数体、注释和行序。

In [3]:
%%project_module satellite_rl.power_refinement.config
"""Validated task controls for frozen-policy, fixed-slot power refinement."""

from __future__ import annotations

from copy import deepcopy
import math


LEGACY_METHODS = ("none", "discrete", "slsqp")
METHODS = (*LEGACY_METHODS, "sca", "de", "pso", "wmmse")
SI_PRIORITY_METHODS = ("sca", "sca_si_lp", "sca_si_lb")
METHOD_LABELS = {
    "none": "T0原分配", "discrete": "原5 W档位调整",
    "slsqp": "SLSQP含需求匹配初始化", "sca": "SCA信赖域凸近似",
    "de": "差分进化DE", "pso": "粒子群PSO",
    "wmmse": "WMMSE速率提议与双指标验收",
    "sca_si_lp": "B：从A继续优先提高SI",
    "sca_si_lb": "C：改进功率估计并优先提高SI",
}
TOTAL_POWER_W = 3000.0
DEFAULTS = {
    "methods": list(METHODS),
    "action_mode": "stochastic",
    "action_seed": 53001,
    "repeats": 3,
    "power_budget_mode": "total",
    "total_power_w": TOTAL_POWER_W,
    "si_tolerance": 1e-8,
    "sgm_tolerance": 1e-8,
    "protect_basic_service": True,
    "maxiter": 300,
    "maxeval": 2000,
    "max_time_seconds": 30.0,
    "discrete_step_w": 5.0,
    "optimizer_seed": 63001,
    "population_size": 24,
    "de_mutation": 0.7,
    "de_crossover": 0.8,
    "pso_inertia": 0.7,
    "pso_cognitive": 1.4,
    "pso_social": 1.4,
    "si_stall_window": 200,
    "si_stall_tolerance": 1e-6,
    "si_min_evaluations": 200,
    "si_max_fraction": 0.85,
    "service_margin_ratio": 5e-9,
}


def default_config():
    return deepcopy(DEFAULTS)


def _integer(value, name, minimum, maximum):
    if type(value) is not int or not minimum <= value <= maximum:
        raise ValueError(f"{name}须为{minimum}–{maximum}的整数")
    return value


def _number(value, name, minimum, maximum):
    if type(value) not in (int, float) or not math.isfinite(value) or not minimum <= value <= maximum:
        raise ValueError(f"{name}须为{minimum}–{maximum}的有限数值")
    return float(value)


def validate_config(raw):
    """Reject unknown or misleading bounds; continuous powers have no per-beam cap."""
    if not isinstance(raw, dict):
        raise ValueError("功率细调配置须为对象")
    unknown = set(raw) - set(DEFAULTS)
    if unknown:
        raise ValueError("未知功率细调参数：" + ", ".join(sorted(unknown)))
    result = default_config()
    result.update(deepcopy(raw))
    if result["methods"] not in (list(LEGACY_METHODS), list(METHODS), list(SI_PRIORITY_METHODS)):
        raise ValueError("请选择原三方法、完整七路线或SI优先三组，方法顺序须一致")
    if result["action_mode"] not in ("deterministic", "stochastic"):
        raise ValueError("动作方式须为确定性或随机采样")
    result["action_seed"] = _integer(result["action_seed"], "动作种子", 0, 2**32 - 1)
    result["repeats"] = _integer(result["repeats"], "每场景动作次数", 1, 30)
    if result["power_budget_mode"] not in ("total", "initial"):
        raise ValueError("功率预算口径须为总预算或初始总功率")
    result["total_power_w"] = _number(result["total_power_w"], "总功率预算/W", TOTAL_POWER_W, TOTAL_POWER_W)
    result["si_tolerance"] = _number(result["si_tolerance"], "SI数值容差", 0, 1e-3)
    result["sgm_tolerance"] = _number(result["sgm_tolerance"], "SGM数值容差", 0, 1e-3)
    if result["protect_basic_service"] is not True:
        raise ValueError("首轮必须开启原基本服务保护")
    result["maxiter"] = _integer(result["maxiter"], "最大迭代次数", 1, 10000)
    result["maxeval"] = _integer(result["maxeval"], "最大容量计算次数", 1, 1000000)
    result["max_time_seconds"] = _number(result["max_time_seconds"], "每次优化时限/秒", 0.01, 3600)
    result["discrete_step_w"] = _number(result["discrete_step_w"], "原功率档位步长/W", 5, 5)
    result["optimizer_seed"] = _integer(result["optimizer_seed"], "优化器基础种子", 0, 2**32 - 1)
    result["population_size"] = _integer(result["population_size"], "种群/粒子数", 4, 256)
    result["de_mutation"] = _number(result["de_mutation"], "DE变异系数", 1e-6, 2)
    result["de_crossover"] = _number(result["de_crossover"], "DE交叉概率", 0, 1)
    result["pso_inertia"] = _number(result["pso_inertia"], "PSO惯性", 0, 0.999999)
    result["pso_cognitive"] = _number(result["pso_cognitive"], "PSO个体系数", 0, 4)
    result["pso_social"] = _number(result["pso_social"], "PSO群体系数", 0, 4)
    result["si_stall_window"] = _integer(result["si_stall_window"], "SI停滞窗口", 1, 10000)
    result["si_stall_tolerance"] = _number(result["si_stall_tolerance"], "SI停滞阈值", 0, 1)
    result["si_min_evaluations"] = _integer(result["si_min_evaluations"], "SI最低容量调用", 1, 1000000)
    result["si_max_fraction"] = _number(result["si_max_fraction"], "SI阶段最大占比", 0.01, 0.99)
    result["service_margin_ratio"] = _number(result["service_margin_ratio"], "基本服务安全裕量", 0, 1e-3)
    if result["methods"] == list(SI_PRIORITY_METHODS):
        fixed = {"si_tolerance": 1e-8, "sgm_tolerance": 1e-8,
                 "maxiter": 300, "maxeval": 2000,
                 "max_time_seconds": 30.0, "si_stall_window": 200,
                 "si_stall_tolerance": 1e-6, "si_min_evaluations": 200,
                 "si_max_fraction": 0.85, "service_margin_ratio": 5e-9}
        if any(result[key] != value for key, value in fixed.items()):
            raise ValueError("SI优先三组须使用方案04冻结的共同容差、阶段规则与追加预算")
    return result


src/satellite_rl/power_refinement/config.py 行数 112 SHA256 97df14c67c08ae8143e289e20ad10f17a428559dc14875de1bac7d9ae844d428


## 2. 固定频率槽下的真实容量、SI/SGM 与导数

`FixedAllocationPhysics.from_scene` 读取每束起始槽/槽数及场景，构造每槽期望信号、重叠干扰和热噪声。`evaluate` 以 **整景终态全部波束** 计算容量 `BW·log2(1+SINR)`；`SI=mean(min(C/D,1))`，SGM 由 `_sgm` 给出。带 `with_jacobian=True` 时返回容量、SI、SGM 对各束功率的导数；后续求解用同一物理函数验收候选。空槽束在 `evaluate` 中必须保持零功率。

以下代码格为冻结 `src/satellite_rl/power_refinement/physics.py` 的全部源码，保留原函数体、注释和行序。

In [4]:
%%project_module satellite_rl.power_refinement.physics
"""Frozen-slot physical capacity and terminal service metrics.

Power is whole-beam watts. The formulas mirror ``envs.satellite``: desired
signal uses the allocated interval's centre, whereas cross-beam interference
uses each overlapping slot centre. The five dB SINR loss applies to the signal.
"""

from dataclasses import dataclass

import numpy as np


_C = 2.99792458e8
_K = 1.380649e-23
_BW = 2.5e7
_F0 = 1.77e10
_LOSS = 10 ** (-5 / 10)


def _pattern(gain_db, angle_deg):
    """The legacy antenna main lobe and Gaussian continuation in dBi."""
    width = 0.5
    sigma = width / (2 * np.sqrt(2 * np.log10(2)))
    angle = np.abs(angle_deg)
    return np.where(
        angle <= width / 2,
        gain_db - 3 * (angle / (width / 2)) ** 2,
        gain_db - 3 - 10 / np.log(10) * (angle ** 2 - (width / 2) ** 2)
        / (2 * sigma ** 2),
    )


def _directions(locations, profile):
    if profile == "europe_v1":
        longitude, earth, orbit = 13.0, 6371.0, 42164.0
    elif profile == "china_legacy":
        longitude, earth, orbit = 122.2, 6371.0, 42164.0
    else:
        raise ValueError(f"Unsupported physical profile: {profile!r}")
    lat, lon = np.deg2rad(locations).T
    ground = earth * np.column_stack((np.cos(lat) * np.cos(lon),
                                      np.cos(lat) * np.sin(lon), np.sin(lat)))
    satellite = np.array([orbit * np.cos(np.deg2rad(longitude)),
                          orbit * np.sin(np.deg2rad(longitude)), 0.0])
    vector = ground - satellite
    return vector / np.linalg.norm(vector, axis=1)[:, None]


def _sgm(capacity, demand):
    """Original SGM and its derivative with respect to capacity (bps)."""
    result = np.zeros_like(capacity)
    derivative = np.zeros_like(capacity)
    beta = float(demand.mean())
    positive = capacity > 0
    with np.errstate(over="ignore", divide="ignore", invalid="ignore", under="ignore"):
        c, d = capacity[positive], demand[positive]
        ratio = c / d
        real = np.where(ratio >= 1, ratio - 1, 1 - 1 / ratio)
        real_prime = np.where(ratio >= 1, 1 / d, d / (c * c))
        imag = (c - d) / beta
        magnitude = np.hypot(real, imag)
        decay = np.exp(-magnitude)
        result[positive] = 1 - (1 - decay) ** 3
        slope_m = -3 * (1 - decay) ** 2 * decay
        slope_c = np.divide(real * real_prime + imag / beta, magnitude,
                            out=np.zeros_like(c), where=magnitude > 0)
        derivative[positive] = slope_m * slope_c
    # The right derivative at zero and derivative at exact demand are zero.
    return np.nan_to_num(result, nan=0.0), np.nan_to_num(derivative, nan=0.0)


@dataclass(frozen=True)
class PhysicalEvaluation:
    capacity_bps: np.ndarray
    si: float
    sgm: float
    beam_si: np.ndarray
    beam_sgm: np.ndarray
    capacity_jacobian: np.ndarray | None = None
    si_jacobian: np.ndarray | None = None
    sgm_jacobian: np.ndarray | None = None


class FixedAllocationPhysics:
    """Vectorized fixed-slot link model for a complete frozen scene."""

    def __init__(self, demand_bps, groups, starts, slots, signal, interference,
                 noise, slot_owner, power_budget_w):
        self.demand_bps = demand_bps
        self.groups = groups
        self.starts = starts
        self.slots = slots
        self.active = slots > 0
        self.signal = signal
        self.interference = interference
        self.noise = noise
        self.slot_owner = slot_owner
        self.beam_count = len(demand_bps)
        self.power_budget_w = power_budget_w

    @classmethod
    def from_scene(cls, scene, config, beams):
        """Freeze actual actions, rejecting requests or incomplete action logs."""
        n = len(beams)
        if n == 0 or set(row.get("beam_id") for row in beams) != set(range(n)):
            raise ValueError("beams must contain one actual allocation per beam")
        rows = sorted(beams, key=lambda row: row["beam_id"])
        def field(name, shape):
            value = np.asarray(scene[name], dtype=np.float64)
            if value.shape != shape or not np.isfinite(value).all():
                raise ValueError(f"Invalid scene field: {name}")
            return value
        locations = field("beam_location", (n, 2))
        tx = field("beam_G_Tx_max", (n, 1)).reshape(-1)
        rx = field("UE_G_Rx_max", (n, 1)).reshape(-1)
        distance = field("distances", (n,)).reshape(-1) * 1000
        demand = field("rate", (n, 1)).reshape(-1)
        temperature = field("noise_temperature", (n, 1)).reshape(-1)
        if np.any(distance <= 0) or np.any(demand <= 0) or np.any(temperature <= 0):
            raise ValueError("Distance, demand, and noise temperature must be positive")
        groups, starts, slots = [], [], []
        for i, row in enumerate(rows):
            group, start, count = row.get("group"), row.get("freq"), row.get("slots")
            actual_power = row.get("power_w")
            if (not isinstance(group, (int, np.integer)) or group != i % 8
                    or not isinstance(count, (int, np.integer)) or not 0 <= count <= 10
                    or (count > 0 and (not isinstance(start, (int, np.integer))
                                       or not 0 <= start <= 100 - count))
                    or not isinstance(actual_power, (int, float, np.number))
                    or not np.isfinite(actual_power) or actual_power < 0
                    or (count == 0 and actual_power != 0)):
                raise ValueError(f"Invalid actual allocation for beam {i}")
            groups.append(int(group))
            starts.append(int(start) if count > 0 else 0)
            slots.append(int(count))
        groups = np.asarray(groups)
        starts = np.asarray(starts)
        slots = np.asarray(slots)
        for i in range(n):
            if not slots[i]:
                continue
            for j in range(i):
                if groups[i] == groups[j] and slots[j] and (
                    max(starts[i], starts[j]) < min(starts[i] + slots[i], starts[j] + slots[j])
                ):
                    raise ValueError("Actual allocations overlap within one group")
        profile = config.get("environment", {}).get("profile", "china_legacy")
        direction = _directions(locations, profile)
        cosines = np.clip(direction @ direction.T, -1, 1)
        angles = np.rad2deg(np.arccos(cosines))
        # Every row denotes one occupied receiver slot; signal has a single
        # interval-centre coefficient, interference has slot-centre coefficients.
        owner = np.repeat(np.arange(n), slots)
        occupied = np.concatenate([np.arange(starts[i], starts[i] + slots[i])
                                   for i in range(n)])
        centre = _F0 + (starts + slots / 2) * _BW
        signal_beam = np.zeros(n)
        active = slots > 0
        signal_beam[active] = (_LOSS * 10 ** ((tx[active] + rx[active]) / 10)
                               * (_C / (4 * np.pi * distance[active] * centre[active])) ** 2
                               / slots[active])
        signal = signal_beam[owner]
        slot_frequency = _F0 + (occupied + 0.5) * _BW
        path = (_C / (4 * np.pi * distance[owner] * slot_frequency)) ** 2
        gain = 10 ** ((_pattern(tx[None, :], angles[owner, :])
                       + rx[owner, None]) / 10)
        overlap = ((groups[owner, None] % 2 == groups[None, :] % 2)
                   & (groups[owner, None] != groups[None, :])
                   & (occupied[:, None] >= starts[None, :])
                   & (occupied[:, None] < starts[None, :] + slots[None, :]))
        interference = np.zeros_like(gain)
        interference[:, active] = gain[:, active] * path[:, None] / slots[None, active]
        interference *= overlap
        noise = _K * temperature[owner] * _BW
        budget = float(config.get("environment", {}).get("power_budget_w", 3000.0))
        if not np.isfinite(budget) or not 0 < budget <= 3000:
            raise ValueError("Invalid power budget")
        return cls(demand, groups, starts, slots, signal, interference, noise,
                   owner, budget)

    def evaluate(self, power_w, with_jacobian=False):
        power = np.asarray(power_w, dtype=np.float64)
        if (power.shape != (self.beam_count,) or not np.isfinite(power).all()
                or (power < 0).any() or (power[~self.active] != 0).any()):
            raise ValueError("Power must be finite, nonnegative, and zero for empty slots")
        denominator = self.noise + self.interference @ power
        desired = self.signal * power[self.slot_owner]
        capacity_slot = _BW / np.log(2) * np.log1p(desired / denominator)
        capacity = np.bincount(self.slot_owner, weights=capacity_slot,
                               minlength=self.beam_count)
        beam_si = np.minimum(capacity / self.demand_bps, 1.0)
        beam_sgm, sgm_prime = _sgm(capacity, self.demand_bps)
        if not with_jacobian:
            return PhysicalEvaluation(capacity, float(beam_si.mean()),
                                      float(beam_sgm.mean()), beam_si, beam_sgm)
        row_jac = (-_BW / np.log(2) * desired[:, None] * self.interference
                   / (denominator[:, None] * (denominator + desired)[:, None]))
        row_jac[np.arange(len(self.slot_owner)), self.slot_owner] += (
            _BW / np.log(2) * self.signal / (denominator + desired))
        jac = np.zeros((self.beam_count, self.beam_count))
        np.add.at(jac, self.slot_owner, row_jac)
        si_prime = np.where(capacity < self.demand_bps, 1 / self.demand_bps, 0)
        return PhysicalEvaluation(capacity, float(beam_si.mean()),
                                  float(beam_sgm.mean()), beam_si, beam_sgm,
                                  jac, si_prime @ jac / self.beam_count,
                                  sgm_prime @ jac / self.beam_count)


src/satellite_rl/power_refinement/physics.py 行数 206 SHA256 715ee1f9b4d9a8efcd3a13ac25077be1ad7374da511d455be7e09b330180ab46


## 3. 优化入口与旧离散/SLSQP 对照

`optimize_power` 在 `method in {"sca", "wmmse"}` 时委托给同版本 `model_based.optimize_model_based`。本格其余较长实现是旧 `none/discrete/slsqp` 的完整代码，不要误认成 A 的求解循环；它说明原离散基线才有 5–50 W/5 W 档位约束。

以下代码格为冻结 `src/satellite_rl/power_refinement/optimizer.py` 的全部源码，保留原函数体、注释和行序。

In [5]:
%%project_module satellite_rl.power_refinement.optimizer
"""Finite-budget, local fixed-slot power refinement with feasible fallback."""

from time import perf_counter

import numpy as np

from .physics import FixedAllocationPhysics


def _metrics(problem, power, evaluation):
    capacity = evaluation.capacity_bps
    return {
        "final_si_mean": evaluation.si,
        "final_sgm_mean": evaluation.sgm,
        "basic_met_count": int(np.count_nonzero(capacity >= 0.1 * problem.demand_bps)),
        "power_w": float(power.sum()),
        "required_bps": problem.demand_bps.tolist(),
        "capacity_bps": capacity.tolist(),
        "effective_served_bps": np.minimum(capacity, problem.demand_bps).tolist(),
        "capacity_demand_ratio": (capacity / problem.demand_bps).tolist(),
        "beam_si": evaluation.beam_si.tolist(),
        "beam_sgm": evaluation.beam_sgm.tolist(),
    }


class _LimitReached(Exception):
    pass


class _WarmBudget(Exception):
    pass


def optimize_power(problem: FixedAllocationPhysics, initial_power, method="none",
                   options=None, callback=None, stop_requested=None):
    """Improve SI first, then SGM, retaining only true feasible candidates.

    ``method`` is ``none``, ``discrete`` (original 5 W moves), or ``slsqp``
    with a bounded demand-matching warm start between its two local phases.
    Neither solver status nor output implies a global optimum.
    ``callback(entry)`` runs for each accepted improvement.
    """
    if method in {"sca", "wmmse"}:
        from .model_based import optimize_model_based
        return optimize_model_based(problem, initial_power, method, options,
                                    callback=callback, stop_requested=stop_requested)
    if method in {"de", "pso"}:
        from .population import optimize_population
        return optimize_population(problem, initial_power, method, options,
                                   callback=callback, stop_requested=stop_requested)
    if not isinstance(problem, FixedAllocationPhysics):
        raise TypeError("problem must be FixedAllocationPhysics")
    if method not in {"none", "discrete", "slsqp"}:
        raise ValueError(f"Unknown method: {method}")
    opts = dict(options or {})
    allowed = {"power_cap_w", "maxiter", "max_evaluations", "maxeval",
               "time_limit_seconds", "max_time_seconds", "ftol", "si_tolerance",
               "sgm_tolerance", "constraint_tolerance", "protect_basic_service",
               "discrete_step_w"}
    if set(opts) - allowed:
        raise ValueError(f"Unknown optimization options: {sorted(set(opts) - allowed)}")
    cap = float(opts.get("power_cap_w", problem.power_budget_w))
    maxiter = int(opts.get("maxiter", 30))
    max_evaluations = int(opts.get("max_evaluations", opts.get("maxeval", 5000)))
    time_limit = float(opts.get("time_limit_seconds", opts.get("max_time_seconds", 60.0)))
    ftol = float(opts.get("ftol", 1e-8))
    si_tolerance = float(opts.get("si_tolerance", 0.0))
    sgm_tolerance = float(opts.get("sgm_tolerance", 0.0))
    feasibility_tolerance = float(opts.get("constraint_tolerance", 1e-8))
    if (opts.get("protect_basic_service", True) is not True
            or float(opts.get("discrete_step_w", 5.0)) != 5.0):
        raise ValueError("Basic service protection and original 5 W grid are fixed")
    if (not 0 < cap <= min(3000.0, problem.power_budget_w)
            or maxiter < 1 or max_evaluations < 1 or time_limit <= 0
            or ftol <= 0 or si_tolerance < 0 or sgm_tolerance < 0
            or feasibility_tolerance < 0
            or not np.isfinite([cap, time_limit, ftol, si_tolerance,
                                  sgm_tolerance, feasibility_tolerance]).all()):
        raise ValueError("Invalid optimization limits or power cap")
    initial = np.asarray(initial_power, dtype=np.float64)
    initial_eval = problem.evaluate(initial, with_jacobian=False)
    if initial.sum() > cap + 1e-7:
        raise ValueError("Initial power exceeds selected budget")
    start_time = perf_counter()
    floor_ratio = np.minimum(initial_eval.capacity_bps / problem.demand_bps, 0.1)
    baseline_sgm = initial_eval.sgm
    best_power = initial.copy()
    best_eval = initial_eval
    count = 1
    full_evaluations = 1
    scalar_evaluations = 0
    history = [{"phase": "initial", "evaluation": count,
                "final_si_mean": initial_eval.si, "final_sgm_mean": initial_eval.sgm,
                "power_w": float(initial.sum()), "elapsed_seconds": 0.0}]
    solver_status = []
    termination = "unchanged" if method == "none" else "completed"
    phase_limit = None
    phase_time_limit = None
    discrete_completed = False
    discrete_exhausted = False
    power_scale = None
    phase2_si_floor = None
    warm_start_status = None

    def check_limit():
        if stop_requested is not None and stop_requested():
            raise _LimitReached("stop_requested")
        if perf_counter() - start_time >= time_limit:
            raise _LimitReached("time_limit")
        if count >= max_evaluations:
            raise _LimitReached("evaluation_limit")
        if phase_limit is not None and count >= phase_limit:
            raise _LimitReached("phase_evaluation_limit")
        if phase_time_limit is not None and perf_counter() - start_time >= phase_time_limit:
            raise _LimitReached("phase_time_limit")

    def feasible(power, ev, si_floor=None):
        return (np.isfinite(power).all() and np.all(power >= 0)
                and np.all(power[~problem.active] == 0)
                and power.sum() <= cap + 1e-7
                and ev.sgm >= baseline_sgm - sgm_tolerance - 1e-10
                and np.all(ev.capacity_bps / problem.demand_bps
                           >= floor_ratio - feasibility_tolerance)
                and (si_floor is None or ev.si >= si_floor - 1e-10))

    def evaluate(power, jac=False):
        nonlocal count, full_evaluations
        check_limit()
        # SLSQP can present tiny negative bound-rounding errors. Its search
        # point remains unmodified; only the physical evaluation is projected.
        safe = np.maximum(np.asarray(power, dtype=np.float64), 0.0)
        safe[~problem.active] = 0.0
        ev = problem.evaluate(safe, with_jacobian=jac)
        count += 1
        full_evaluations += 1
        return ev

    def retain(power, ev, phase, si_floor=None):
        nonlocal best_power, best_eval
        if not feasible(power, ev, si_floor):
            return
        if phase == "si":
            improved = (ev.si > best_eval.si + 1e-10
                        or (abs(ev.si - best_eval.si) <= 1e-10
                            and ev.sgm > best_eval.sgm + 1e-10))
        else:
            improved = (ev.sgm > best_eval.sgm + 1e-10
                        or (abs(ev.sgm - best_eval.sgm) <= 1e-10
                            and ev.si > best_eval.si + 1e-10))
        if improved:
            best_power, best_eval = np.array(power, copy=True), ev
            entry = {"phase": phase, "evaluation": count,
                     "final_si_mean": ev.si, "final_sgm_mean": ev.sgm,
                     "power_w": float(power.sum()),
                     "elapsed_seconds": perf_counter() - start_time}
            history.append(entry)
            if callback is not None:
                callback(dict(entry))

    def demand_matching_warm_start():
        """Reduce oversupplied beam power with frozen current interference.

        Each scalar capacity computation consumes the same one-call budget as
        a full evaluation. Intermediate points may be infeasible for SGM; only
        independently checked full-scene points are retained. The final SLSQP
        starts from the best retained feasible point, never an unchecked one.
        """
        nonlocal count, scalar_evaluations, warm_start_status
        started = perf_counter()
        phase_floor = best_eval.si - si_tolerance
        working = best_power.copy()
        scalar_start = scalar_evaluations
        full_start = full_evaluations
        retained_start = len(history)
        warm_budget = min(800, int(max_evaluations * 0.4))
        reserve = max(2, int(max_evaluations * 0.3))
        ceiling = min(max_evaluations - reserve, count + warm_budget)
        changed = 0
        sweeps = 0

        def scalar_capacity(owner, watts):
            nonlocal count, scalar_evaluations
            if (count >= ceiling or perf_counter() - start_time >= 0.7 * time_limit):
                raise _WarmBudget
            check_limit()
            slot_rows = problem.slot_owner == owner
            denominator = (problem.noise[slot_rows]
                           + problem.interference[slot_rows] @ working)
            signal = problem.signal[slot_rows] * watts
            value = float((2.5e7 / np.log(2)
                           * np.log1p(signal / denominator)).sum())
            count += 1
            scalar_evaluations += 1
            return value

        for sweep in range(2):
            sweep_changed = 0
            for owner in np.flatnonzero(problem.active):
                if count + 10 >= ceiling or perf_counter() - start_time >= 0.7 * time_limit:
                    break
                current = working[owner]
                if current <= 0:
                    continue
                try:
                    present = scalar_capacity(owner, current)
                    demand = problem.demand_bps[owner]
                    if present <= demand * (1 + 1e-5):
                        continue
                    low, high = 0.0, current
                    for _ in range(8):
                        middle = (low + high) / 2
                        if scalar_capacity(owner, middle) >= demand:
                            high = middle
                        else:
                            low = middle
                    if current - high <= 1e-7:
                        continue
                    working[owner] = high
                    ev = evaluate(working)
                    retain(working, ev, "demand_matching_warm_start", phase_floor)
                    changed += 1
                    sweep_changed += 1
                except _WarmBudget:
                    break
            sweeps = sweep + 1
            if not sweep_changed or count + 10 >= ceiling:
                break
        warm_start_status = {
            "component": "demand_matching_warm_start",
            "scalar_capacity_evaluations": scalar_evaluations - scalar_start,
            "full_scene_evaluations": full_evaluations - full_start,
            "changed_beams": changed, "sweeps": sweeps,
            "accepted_candidates": len(history) - retained_start,
            "elapsed_seconds": perf_counter() - started,
            "stopped_by_reservation": count + 10 >= ceiling,
        }

    try:
        if method == "discrete":
            # Input model actions are 5 W multiples. Do not silently quantize.
            active = np.flatnonzero(problem.active)
            if not np.allclose(initial[active] / 5, np.rint(initial[active] / 5),
                               atol=1e-7, rtol=0):
                raise ValueError("Discrete baseline requires original 5 W power levels")
            if np.any(initial[active] < 5 - 1e-7) or np.any(initial[active] > 50 + 1e-7):
                raise ValueError("Discrete baseline requires active beams in 5–50 W")
            for phase in ("si", "sgm"):
                si_floor = None if phase == "si" else best_eval.si - si_tolerance
                if phase == "sgm":
                    phase2_si_floor = si_floor
                for _ in range(maxiter):
                    check_limit()
                    old = best_power.copy()
                    # Single-beam increments/decrements, then 5 W transfers.
                    moves = []
                    for i in active:
                        if best_power[i] >= 10 - 1e-8:
                            moves.append((i, -1, 0))
                        if best_power[i] <= 45 + 1e-8:
                            moves.append((i, -1, 1))
                    for i in active:
                        if best_power[i] < 10 - 1e-8:
                            continue
                        for j in active:
                            if i != j and best_power[j] <= 45 + 1e-8:
                                moves.append((i, j, 2))
                    selected = None
                    selected_ev = best_eval
                    try:
                        for i, j, kind in moves:
                            candidate = old.copy()
                            if kind == 0:
                                candidate[i] -= 5
                            elif kind == 1:
                                candidate[i] += 5
                            else:
                                candidate[i] -= 5
                                candidate[j] += 5
                            if candidate.sum() > cap + 1e-7:
                                continue
                            ev = evaluate(candidate)
                            if not feasible(candidate, ev, si_floor):
                                continue
                            if phase == "si":
                                key, current = (ev.si, ev.sgm, -candidate.sum()), (
                                    selected_ev.si, selected_ev.sgm,
                                    -selected.sum() if selected is not None else -old.sum())
                            else:
                                key, current = (ev.sgm, ev.si, -candidate.sum()), (
                                    selected_ev.sgm, selected_ev.si,
                                    -selected.sum() if selected is not None else -old.sum())
                            if key > current and (key[0] > current[0] + 1e-10
                                                  or key[1] > current[1] + 1e-10):
                                selected, selected_ev = candidate, ev
                    except _LimitReached:
                        if selected is not None:
                            retain(selected, selected_ev, phase, si_floor)
                        raise
                    if selected is None:
                        break
                    retain(selected, selected_ev, phase, si_floor)
                    if np.array_equal(old, best_power):
                        break
                else:
                    discrete_exhausted = True
            discrete_completed = not discrete_exhausted
        elif method == "slsqp":
            try:
                from scipy.optimize import minimize
            except ImportError as exc:
                raise RuntimeError("SLSQP requires scipy") from exc
            active = np.flatnonzero(problem.active)
            if len(active):
                demand = problem.demand_bps
                n_active = len(active)
                # Dimensionless p/scale makes power and auxiliary SI variables
                # comparable for SLSQP's line search and stopping tests.
                power_scale = cap / n_active
                for phase in ("si", "sgm"):
                    phase_limit = (max(2, min(max_evaluations - 1,
                                              int(max_evaluations * 0.65)))
                                   if phase == "si" and max_evaluations > 2 else None)
                    phase_time_limit = 0.65 * time_limit if phase == "si" else None
                    check_limit()
                    phase_start = best_power.copy()
                    phase_si_floor = best_eval.si - si_tolerance if phase == "sgm" else None
                    if phase == "sgm":
                        phase2_si_floor = phase_si_floor
                    cache_x = None
                    cache_ev = None

                    def evaluated_x(x):
                        nonlocal cache_x, cache_ev
                        p = np.zeros(problem.beam_count)
                        p[active] = x[:n_active] * power_scale
                        if cache_x is None or not np.array_equal(x, cache_x):
                            cache_ev = evaluate(p, jac=True)
                            cache_x = np.array(x, copy=True)
                            retain(p, cache_ev, phase, phase_si_floor)
                        return cache_ev

                    def capacity_constraint(x):
                        ev = evaluated_x(x)
                        return ev.capacity_bps / demand - x[n_active:]

                    def capacity_jac(x):
                        ev = evaluated_x(x)
                        return np.column_stack((ev.capacity_jacobian[:, active] * power_scale
                                                / demand[:, None],
                                                -np.eye(problem.beam_count)))

                    def floor_constraint(x):
                        return evaluated_x(x).capacity_bps / demand - floor_ratio

                    def floor_jac(x):
                        return (evaluated_x(x).capacity_jacobian[:, active] * power_scale
                                / demand[:, None])

                    def sgm_constraint(x):
                        return evaluated_x(x).sgm - baseline_sgm

                    def sgm_jac(x):
                        return evaluated_x(x).sgm_jacobian[active] * power_scale

                    if phase == "si":
                        u0 = np.minimum(best_eval.beam_si, 1)
                        x0 = np.concatenate((phase_start[active] / power_scale, u0))
                        constraints = [
                            {"type": "ineq", "fun": lambda x: cap / power_scale - x[:n_active].sum(),
                             "jac": lambda x: np.r_[-np.ones(n_active),
                                                        np.zeros(problem.beam_count)]},
                            {"type": "ineq", "fun": capacity_constraint,
                             "jac": capacity_jac},
                            {"type": "ineq", "fun": floor_constraint,
                             "jac": lambda x: np.column_stack((floor_jac(x),
                                                              np.zeros((problem.beam_count,
                                                                        problem.beam_count))))},
                            {"type": "ineq", "fun": sgm_constraint,
                             "jac": lambda x: np.r_[sgm_jac(x),
                                                      np.zeros(problem.beam_count)]},
                        ]
                        objective = lambda x: -float(np.mean(x[n_active:]))
                        objective_jac = lambda x: np.r_[np.zeros(n_active),
                                                         -np.ones(problem.beam_count) / problem.beam_count]
                        bounds = [(0, None)] * n_active + [(0, 1)] * problem.beam_count
                    else:
                        # Keep the SI epigraph in phase two as well. Direct
                        # min(C/D, 1) has a kink at demand and a zero gradient
                        # once every beam is capped, obscuring feasible SGM
                        # directions along the SI=1 boundary.
                        u0 = np.minimum(best_eval.beam_si, 1)
                        x0 = np.concatenate((phase_start[active] / power_scale, u0))
                        constraints = [
                            {"type": "ineq", "fun": lambda x: cap / power_scale - x[:n_active].sum(),
                             "jac": lambda x: np.r_[-np.ones(n_active),
                                                        np.zeros(problem.beam_count)]},
                            {"type": "ineq", "fun": capacity_constraint,
                             "jac": capacity_jac},
                            {"type": "ineq", "fun": floor_constraint,
                             "jac": lambda x: np.column_stack((floor_jac(x),
                                                              np.zeros((problem.beam_count,
                                                                        problem.beam_count))))},
                            {"type": "ineq", "fun": sgm_constraint,
                             "jac": lambda x: np.r_[sgm_jac(x),
                                                      np.zeros(problem.beam_count)]},
                            {"type": "ineq", "fun": lambda x: np.mean(x[n_active:]) - phase_si_floor,
                             "jac": lambda x: np.r_[np.zeros(n_active),
                                                      np.ones(problem.beam_count) / problem.beam_count]},
                        ]
                        objective = lambda x: -evaluated_x(x).sgm
                        objective_jac = lambda x: np.r_[-sgm_jac(x),
                                                          np.zeros(problem.beam_count)]
                        bounds = [(0, None)] * n_active + [(0, 1)] * problem.beam_count
                    try:
                        result = minimize(objective, x0, method="SLSQP", jac=objective_jac,
                                          bounds=bounds, constraints=constraints,
                                          options={"maxiter": maxiter, "ftol": ftol, "disp": False})
                    except _LimitReached as exc:
                        solver_status.append({"phase": phase, "success": False,
                                              "message": str(exc), "iterations": None})
                        if phase == "si" and str(exc) in {
                                "phase_evaluation_limit", "phase_time_limit"}:
                            phase_limit = None
                            phase_time_limit = None
                            demand_matching_warm_start()
                            continue
                        raise
                    solver_status.append({"phase": phase, "success": bool(result.success),
                                          "message": str(result.message),
                                          "iterations": int(result.nit)})
                    phase_limit = None
                    phase_time_limit = None
                    if phase == "si":
                        demand_matching_warm_start()
    except _LimitReached as exc:
        termination = str(exc)
    except Exception:
        # Programmer/data errors are not solver failures and must be visible.
        raise

    if method == "slsqp" and any(not item["success"] for item in solver_status):
        if termination == "completed":
            termination = "solver_nonconvergence"
    if method == "discrete" and not discrete_completed and termination == "completed":
        termination = "iteration_limit"
    if method != "none" and len(history) == 1 and termination == "completed":
        termination = "no_feasible_improvement"
    converged = (method == "none" or
                 (method == "discrete" and discrete_completed and termination in {
                     "completed", "no_feasible_improvement"}) or
                 (method == "slsqp" and
                  (not np.any(problem.active) or
                   len(solver_status) == 2 and all(row["success"] for row in solver_status))
                  and termination in {"completed", "no_feasible_improvement"}))
    error = {
        "budget_excess_w": max(0.0, float(best_power.sum() - cap)),
        "negative_power_w": max(0.0, float(-best_power.min())),
        "empty_slot_power_w": float(np.max(np.abs(best_power[~problem.active])))
        if np.any(~problem.active) else 0.0,
        "sgm_shortfall": max(0.0, baseline_sgm - best_eval.sgm),
        "basic_shortfall_ratio": float(np.maximum(
            floor_ratio - best_eval.capacity_bps / problem.demand_bps, 0).max()),
        "si_shortfall": max(0.0, initial_eval.si - best_eval.si),
        "phase2_si_shortfall": (max(0.0, phase2_si_floor - best_eval.si)
                                if phase2_si_floor is not None else 0.0),
    }
    return {
        "method": method, "power_w": best_power.tolist(),
        "initial_metrics": _metrics(problem, initial, initial_eval),
        "metrics": _metrics(problem, best_power, best_eval),
        "history": history, "evaluations": count,
        "full_scene_evaluations": full_evaluations,
        "scalar_capacity_evaluations": scalar_evaluations,
        "elapsed_seconds": perf_counter() - start_time,
        "constraint_error": error,
        "termination_reason": termination,
        "converged": bool(converged),
        "fallback_used": bool(method != "none" and np.array_equal(best_power, initial)),
        "solver_status": solver_status,
        "solver_power_scale_w": power_scale,
        "warm_start_status": warm_start_status,
        "algorithm_components": (
            ["initial_allocation"] if method == "none" else
            ["discrete_5w_coordinate_and_exchange"] if method == "discrete" else
            ["slsqp_si_epigraph", "demand_matching_warm_start",
             "slsqp_sgm_epigraph"]),
        "power_cap_w": cap,
    }


src/satellite_rl/power_refinement/optimizer.py 行数 488 SHA256 7248e6a2047a957288ba3911eaf2649f8f96d47ce8ec68647d9310d7ca87e293


## 3.1 A：原 SCA 的真实两阶段更新

A 的实际循环在 `optimize_model_based`：`_sca_proposal` 构造信赖域内的容量近似，先提高 SI、再在已得 SI 与基本服务约束下改善 SGM。`valid`、真实 `problem.evaluate`、回溯、容量调用/时间/轮数停止和已验可行解回退都在此模块。连续功率仅非负、整景总和不超过所选预算；空槽功率为 0。B/C 复用这个 A 终值。

以下代码格为冻结 `src/satellite_rl/power_refinement/model_based.py` 的全部源码，保留原函数体、注释和行序。

In [6]:
%%project_module satellite_rl.power_refinement.model_based
"""Model-based fixed-slot power proposals with exact physical acceptance.

SCA solves a trust-region linear convex approximation of the *actual* SI
epigraph and then of SGM. WMMSE solves a weighted sum-rate surrogate on the
slot links. Neither surrogate is asserted to equal the original lexicographic
SI/SGM problem; every retained point is checked with the original physics.
"""

from time import perf_counter

import numpy as np

from .physics import FixedAllocationPhysics, _sgm


def _metrics(problem, power, ev):
    capacity = ev.capacity_bps
    return {
        "final_si_mean": ev.si,
        "final_sgm_mean": ev.sgm,
        "basic_met_count": int(np.count_nonzero(capacity >= 0.1 * problem.demand_bps)),
        "power_w": float(power.sum()),
        "required_bps": problem.demand_bps.tolist(),
        "capacity_bps": capacity.tolist(),
        "effective_served_bps": np.minimum(capacity, problem.demand_bps).tolist(),
        "capacity_demand_ratio": (capacity / problem.demand_bps).tolist(),
        "beam_si": ev.beam_si.tolist(),
        "beam_sgm": ev.beam_sgm.tolist(),
    }


def _wmmse_proposal(problem, power, ev, cap, phase):
    """One exact weighted-MSE block update for a local rate surrogate.

    Each occupied slot is a scalar Gaussian interference link, while all
    slots of a beam share x_i=sqrt(p_i). The receiver and MSE weights have
    closed forms. With those fixed, the x step is a diagonal convex quadratic
    under sum(x_i**2)<=cap, solved through a scalar multiplier.
    """
    active = problem.active
    if phase == "si":
        weights = np.where(ev.capacity_bps < problem.demand_bps * (1 - 1e-8),
                           1 / problem.demand_bps, 0.0)
    else:
        _, derivative = _sgm(ev.capacity_bps, problem.demand_bps)
        weights = np.maximum(derivative, 0)
    if not np.any(weights > 0):
        return None
    weights = weights / weights.max()
    owner = problem.slot_owner
    # Row normalization removes tiny physical channel/noise units without
    # changing each slot's SINR, MMSE receiver or power minimizer.
    signal = problem.signal / problem.noise
    interference = problem.interference / problem.noise[:, None]
    x = np.sqrt(power)
    denominator = 1.0 + interference @ power
    desired = signal * power[owner]
    total = denominator + desired
    receiver = np.sqrt(signal) * x[owner] / total
    mse_weight = total / denominator
    row_weight = weights[owner] * mse_weight
    linear = np.bincount(owner, weights=row_weight * receiver * np.sqrt(signal),
                         minlength=problem.beam_count)
    quadratic = interference.T @ (row_weight * receiver**2)
    quadratic += np.bincount(owner, weights=row_weight * receiver**2 * signal,
                             minlength=problem.beam_count)
    linear[~active] = 0.0
    quadratic[~active] = 0.0
    if not np.any(linear > 0):
        return None

    def candidate(multiplier):
        result = np.divide(linear, quadratic + multiplier,
                           out=np.zeros_like(linear), where=quadratic + multiplier > 0)
        result[~active] = 0
        return result

    trial = candidate(0.0)
    if float(trial @ trial) > cap:
        low, high = 0.0, 1.0
        while float(candidate(high) @ candidate(high)) > cap:
            high *= 2
        for _ in range(50):
            middle = (low + high) / 2
            if float(candidate(middle) @ candidate(middle)) > cap:
                low = middle
            else:
                high = middle
        trial = candidate(high)
    return trial**2


def _sca_proposal(problem, power, ev, cap, phase, si_floor, trust_w,
                  floor_ratio, baseline_sgm, sgm_tolerance, remaining_time):
    """Solve an LP formed by first-order capacity/SGM models in a trust box."""
    from scipy.optimize import linprog

    active = np.flatnonzero(problem.active)
    m, n = len(active), problem.beam_count
    scale = cap / m
    x0 = power[active] / scale
    ratio = ev.capacity_bps / problem.demand_bps
    jac = ev.capacity_jacobian[:, active] * scale / problem.demand_bps[:, None]
    # Variables are scaled whole-beam powers x and epigraph service u.
    # u_i <= C_i^linear/D_i, 0<=u_i<=1.
    capacity_rows = np.column_stack((-jac, np.eye(n)))
    capacity_rhs = ratio - jac @ x0
    floor_rows = np.column_stack((-jac, np.zeros((n, n))))
    floor_rhs = ratio - jac @ x0 - floor_ratio
    grad_sgm = ev.sgm_jacobian[active] * scale
    sgm_row = np.r_[-grad_sgm, np.zeros(n)][None, :]
    sgm_rhs = np.array([ev.sgm - baseline_sgm + sgm_tolerance - grad_sgm @ x0])
    budget_row = np.r_[np.ones(m), np.zeros(n)][None, :]
    rows = [capacity_rows, floor_rows, sgm_row, budget_row]
    rhs = [capacity_rhs, floor_rhs, sgm_rhs, np.array([cap / scale])]
    if phase == "sgm":
        rows.append(np.r_[np.zeros(m), -np.ones(n)][None, :])
        rhs.append(np.array([-n * si_floor]))
        objective = np.r_[-grad_sgm, np.zeros(n)]
    else:
        objective = np.r_[np.zeros(m), -np.ones(n) / n]
    trust = trust_w / scale
    bounds = [(max(0, value - trust), value + trust) for value in x0]
    bounds.extend([(0, 1)] * n)
    result = linprog(objective, A_ub=np.vstack(rows), b_ub=np.concatenate(rhs),
                     bounds=bounds, method="highs",
                     options={"time_limit": max(0.001, remaining_time)})
    if not result.success:
        return None, str(result.message)
    proposed = np.zeros(problem.beam_count)
    proposed[active] = np.maximum(result.x[:m], 0) * scale
    return proposed, str(result.message)


def optimize_model_based(problem: FixedAllocationPhysics, initial_power,
                         method="sca", options=None, callback=None,
                         stop_requested=None):
    """Use model-based proposals within the original two-phase acceptance rule.

    Budget counts one complete physical capacity evaluation per candidate.
    LP and WMMSE surrogate operations are recorded separately and do not count
    as physical calls. A solver result is never returned without true checks.
    """
    if not isinstance(problem, FixedAllocationPhysics):
        raise TypeError("problem must be FixedAllocationPhysics")
    if method not in {"sca", "wmmse"}:
        raise ValueError(f"Unknown model-based method: {method}")
    opts = dict(options or {})
    allowed = {"power_cap_w", "maxiter", "max_evaluations", "maxeval",
               "time_limit_seconds", "max_time_seconds", "ftol", "si_tolerance",
               "sgm_tolerance", "constraint_tolerance", "protect_basic_service",
               "discrete_step_w"}
    if set(opts) - allowed:
        raise ValueError(f"Unknown optimization options: {sorted(set(opts) - allowed)}")
    cap = float(opts.get("power_cap_w", problem.power_budget_w))
    maxiter = int(opts.get("maxiter", 300))
    maxeval = int(opts.get("max_evaluations", opts.get("maxeval", 2000)))
    limit_seconds = float(opts.get("time_limit_seconds", opts.get("max_time_seconds", 30.0)))
    si_tolerance = float(opts.get("si_tolerance", 0.0))
    sgm_tolerance = float(opts.get("sgm_tolerance", 0.0))
    constraint_tolerance = float(opts.get("constraint_tolerance", 1e-8))
    ftol = float(opts.get("ftol", 1e-8))
    if (opts.get("protect_basic_service", True) is not True
            or float(opts.get("discrete_step_w", 5.0)) != 5.0):
        raise ValueError("Basic service protection and original grid control are fixed")
    if (not np.isfinite([cap, limit_seconds, si_tolerance, sgm_tolerance,
                         constraint_tolerance, ftol]).all()
            or not 0 < cap <= min(3000, problem.power_budget_w)
            or maxiter < 1 or maxeval < 1 or limit_seconds <= 0 or ftol <= 0
            or min(si_tolerance, sgm_tolerance, constraint_tolerance) < 0):
        raise ValueError("Invalid optimization limits")
    initial = np.asarray(initial_power, dtype=np.float64)
    original = problem.evaluate(initial, with_jacobian=True)
    if initial.sum() > cap + 1e-7:
        raise ValueError("Initial power exceeds selected budget")
    floor_ratio = np.minimum(original.capacity_bps / problem.demand_bps, 0.1)
    best_power, best_eval = initial.copy(), original
    start = perf_counter()
    evaluations = 1
    proposals = 0
    lp_subproblems = 0
    lp_elapsed_seconds = 0.0
    history = [{"phase": "initial", "evaluation": 1, "final_si_mean": original.si,
                "final_sgm_mean": original.sgm, "power_w": float(initial.sum()),
                "elapsed_seconds": 0.0}]
    solver_status = []
    termination = "completed"
    phase2_si_floor = None

    def timed_out():
        return perf_counter() - start >= limit_seconds

    def check_budget():
        if stop_requested is not None and stop_requested():
            return "stop_requested"
        if timed_out():
            return "time_limit"
        if evaluations >= maxeval:
            return "evaluation_limit"
        return None

    def valid(power, ev, si_floor):
        return (np.isfinite(power).all() and np.all(power >= 0)
                and np.all(power[~problem.active] == 0)
                and power.sum() <= cap + 1e-7
                and ev.sgm >= original.sgm - sgm_tolerance - 1e-10
                and np.all(ev.capacity_bps / problem.demand_bps
                           >= floor_ratio - constraint_tolerance)
                and (si_floor is None or ev.si >= si_floor - 1e-10))

    for phase in ("si", "sgm"):
        phase_start = perf_counter()
        phase_iterations = 0
        accepted = 0
        phase_stop_reason = None
        message = "no proposal"
        if phase == "sgm":
            phase2_si_floor = best_eval.si - si_tolerance
        trust = max(cap / max(1, np.count_nonzero(problem.active)), 1.0)
        trust *= 0.5
        for _ in range(maxiter):
            reason = check_budget()
            if reason:
                termination = reason
                phase_stop_reason = reason
                break
            if phase == "si" and (evaluations >= max(2, int(0.65 * maxeval))
                                  or perf_counter() - start >= 0.65 * limit_seconds):
                phase_stop_reason = "phase_budget"
                break
            if not np.any(problem.active):
                phase_stop_reason = "no_active_beams"
                break
            phase_iterations += 1
            if method == "sca":
                lp_started = perf_counter()
                proposal, message = _sca_proposal(
                    problem, best_power, best_eval, cap, phase, phase2_si_floor,
                    trust, floor_ratio, original.sgm, sgm_tolerance,
                    limit_seconds - (perf_counter() - start))
                lp_subproblems += 1
                lp_elapsed_seconds += perf_counter() - lp_started
            else:
                proposal = _wmmse_proposal(problem, best_power, best_eval, cap, phase)
                message = "weighted-MMSE rate proposal"
            proposals += 1
            if proposal is None:
                if method == "sca" and trust > 1e-5:
                    trust *= 0.5
                    continue
                if method == "sca":
                    phase_stop_reason = "trust_region_small"
                elif phase == "si":
                    phase_stop_reason = ("surrogate_zero_weights" if not np.any(
                        best_eval.capacity_bps < problem.demand_bps * (1 - 1e-8))
                        else "surrogate_zero_gain")
                else:
                    _, derivative = _sgm(best_eval.capacity_bps, problem.demand_bps)
                    phase_stop_reason = ("surrogate_zero_weights" if not np.any(
                        derivative > 0) else "surrogate_zero_gain")
                break
            direction = proposal - best_power
            if np.max(np.abs(direction)) < 1e-9:
                phase_stop_reason = "surrogate_stationary_direction"
                break
            improvement = False
            for backtrack in range(9):
                reason = check_budget()
                if reason:
                    termination = reason
                    phase_stop_reason = reason
                    break
                step = 2.0 ** -backtrack
                candidate = best_power + step * direction
                candidate[~problem.active] = 0
                candidate = np.maximum(candidate, 0)
                ev = problem.evaluate(candidate, with_jacobian=True)
                evaluations += 1
                if not valid(candidate, ev, phase2_si_floor):
                    continue
                if phase == "si":
                    improvement = (ev.si > best_eval.si + 1e-10
                                   or (abs(ev.si - best_eval.si) <= 1e-10
                                       and ev.sgm > best_eval.sgm + 1e-10))
                else:
                    improvement = (ev.sgm > best_eval.sgm + 1e-10
                                   or (abs(ev.sgm - best_eval.sgm) <= 1e-10
                                       and ev.si > best_eval.si + 1e-10))
                if improvement:
                    best_power, best_eval = candidate, ev
                    accepted += 1
                    entry = {"phase": phase, "evaluation": evaluations,
                             "final_si_mean": ev.si, "final_sgm_mean": ev.sgm,
                             "power_w": float(candidate.sum()),
                             "elapsed_seconds": perf_counter() - start,
                             "step_fraction": step}
                    history.append(entry)
                    if callback is not None:
                        callback(dict(entry))
                    break
            if termination != "completed":
                break
            if method == "sca":
                trust = min(cap, trust * 1.4) if improvement else trust * 0.5
                if trust < 1e-5:
                    phase_stop_reason = "trust_region_small"
                    break
            elif not improvement:
                phase_stop_reason = "no_acceptable_step"
                break
        if phase_stop_reason is None:
            phase_stop_reason = "iteration_limit"
        solver_status.append({"phase": phase, "success": accepted > 0,
                              "success_scope": "accepted true feasible improvement only",
                              "stop_reason": phase_stop_reason,
                              "message": message if phase_iterations else "no proposal",
                              "iterations": phase_iterations, "accepted": accepted,
                              "elapsed_seconds": perf_counter() - phase_start})
        if termination != "completed":
            break
        termination = phase_stop_reason
        if phase == "si":
            # A local phase stop still permits the SGM phase; only global
            # stop/time/evaluation limits prevent it.
            termination = "completed"
    if termination == "completed":
        termination = solver_status[-1]["stop_reason"]
    error = {
        "budget_excess_w": max(0.0, float(best_power.sum() - cap)),
        "negative_power_w": max(0.0, float(-best_power.min())),
        "empty_slot_power_w": (float(np.max(np.abs(best_power[~problem.active])))
                               if np.any(~problem.active) else 0.0),
        "sgm_shortfall": max(0.0, original.sgm - best_eval.sgm),
        "basic_shortfall_ratio": float(np.maximum(
            floor_ratio - best_eval.capacity_bps / problem.demand_bps, 0).max()),
        "si_shortfall": max(0.0, original.si - best_eval.si),
        "phase2_si_shortfall": (max(0.0, phase2_si_floor - best_eval.si)
                                if phase2_si_floor is not None else 0.0),
    }
    return {
        "method": method, "power_w": best_power.tolist(),
        "initial_metrics": _metrics(problem, initial, original),
        "metrics": _metrics(problem, best_power, best_eval),
        "history": history, "evaluations": evaluations,
        "full_scene_evaluations": evaluations, "scalar_capacity_evaluations": 0,
        "elapsed_seconds": perf_counter() - start, "constraint_error": error,
        "termination_reason": termination, "converged": False,
        "convergence_scope": "original SI/SGM stationary point not certified",
        "fallback_used": bool(np.array_equal(best_power, initial)),
        "solver_status": solver_status,
        "solver_power_scale_w": (cap / np.count_nonzero(problem.active)
                                  if np.any(problem.active) else None),
        "warm_start_status": None,
        "algorithm_components": (["sca_trust_region_lp", "true_metric_backtracking"]
                                 if method == "sca" else
                                 ["slot_weighted_mmse_rate_proposal", "true_metric_backtracking"]),
        "surrogate_proposals": proposals, "power_cap_w": cap,
        "lp_subproblems": lp_subproblems,
        "lp_elapsed_seconds": lp_elapsed_seconds,
        "surrogate_weight_rule": (
            None if method == "sca" else
            "SI: 1/D for beams below demand, zero otherwise; "
            "SGM: positive part of dSGM/dC; weights normalized per step; "
            "weighted sum-rate is a proposal only"),
    }


src/satellite_rl/power_refinement/model_based.py 行数 365 SHA256 c741eafc825f210d33529e83972fd8b08ab90178652e21b4d792f8150e5e3539


## 4. B/C：共用 A 起点，SI 优先两阶段

`_proposal_lp`（B）用容量在当前点的切线和线性规划；`_proposal_lb`（C）保留 log(总接收功率)、只线性化 log(干扰)，形成更紧的凹下界与 Clarabel 子问题。`optimize_priority` 先提高 SI，再把最高可行 SI 锁为门槛改善 SGM；每个提议均用 `problem.evaluate` 的真实终态物理量验收，信赖域/回溯和最高 SI 锚点防止代理模型误导。B/C 使用**同一个 A 终值分别追加**，互不串接。

以下代码格为冻结 `src/satellite_rl/power_refinement/priority.py` 的全部源码，保留原函数体、注释和行序。

In [7]:
%%project_module satellite_rl.power_refinement.priority
"""SI-first refinement from a frozen, already evaluated SCA allocation.

Both variants use the same trust box and true-physics acceptance.  The LP uses
the tangent capacity, while the LB variant retains the concave log(total)
term and only linearizes log(interference).  Neither claims global optimality.
"""

from time import perf_counter

import numpy as np

from .model_based import _metrics
from .physics import FixedAllocationPhysics


_BW_OVER_LN2 = 2.5e7 / np.log(2.0)
_SI_TIE = 1e-10
_SGM_TIE = 1e-10


def _capacity_lower_bound(problem, centre, trial):
    """Beam capacities and gradient of the tight global concave lower bound.

    This NumPy version is used for numerical checks.  The optimization model
    below uses the same expression in CVXPY, with each row divided by noise.
    """
    h = problem.interference / problem.noise[:, None]
    a = problem.signal / problem.noise
    owner = problem.slot_owner
    q0 = 1.0 + h @ centre
    q = 1.0 + h @ trial
    total = q + a * trial[owner]
    lower_slot = _BW_OVER_LN2 * (np.log(total) - np.log(q0) - (q - q0) / q0)
    slot_grad = _BW_OVER_LN2 * (h / total[:, None] - h / q0[:, None])
    slot_grad[np.arange(len(owner)), owner] += _BW_OVER_LN2 * a / total
    cap = np.bincount(owner, weights=lower_slot, minlength=problem.beam_count)
    jac = np.zeros((problem.beam_count, problem.beam_count))
    np.add.at(jac, owner, slot_grad)
    return cap, jac


def _proposal_lp(problem, power, ev, cap, phase, anchor_si, floor_ratio,
                 trust_w, remaining_time):
    from scipy.optimize import linprog

    active = np.flatnonzero(problem.active)
    n, m = problem.beam_count, len(active)
    scale = cap / m
    x0 = power[active] / scale
    ratio = ev.capacity_bps / problem.demand_bps
    jac = ev.capacity_jacobian[:, active] * scale / problem.demand_bps[:, None]
    rows = [np.column_stack((-jac, np.eye(n))),
            np.column_stack((-jac, np.zeros((n, n)))),
            np.r_[np.ones(m), np.zeros(n)][None, :]]
    rhs = [ratio - jac @ x0,
           ratio - jac @ x0 - floor_ratio,
           np.array([cap / scale])]
    if phase == "sgm":
        rows.append(np.r_[np.zeros(m), -np.ones(n)][None, :])
        rhs.append(np.array([-n * (anchor_si - 1e-8)]))
        objective = np.r_[-ev.sgm_jacobian[active] * scale, np.zeros(n)]
    else:
        objective = np.r_[np.zeros(m), -np.ones(n) / n]
    trust = trust_w / scale
    bounds = [(max(0.0, value - trust), value + trust) for value in x0]
    bounds.extend([(0.0, 1.0)] * n)
    result = linprog(objective, A_ub=np.vstack(rows), b_ub=np.concatenate(rhs),
                     bounds=bounds, method="highs",
                     options={"time_limit": max(0.001, remaining_time)})
    if not result.success:
        return None, str(result.message)
    candidate = np.zeros(n)
    candidate[active] = np.maximum(result.x[:m], 0.0) * scale
    return candidate, str(result.message)


def _proposal_lb(problem, power, ev, cap, phase, anchor_si, floor_ratio,
                 trust_w, remaining_time):
    """Solve the log(total)-tangent(log(interference)) concave subproblem."""
    try:
        import cvxpy as cp
    except ImportError as exc:
        raise RuntimeError("sca_si_lb requires cvxpy with Clarabel") from exc

    deadline = perf_counter() + remaining_time
    active = np.flatnonzero(problem.active)
    n, m = problem.beam_count, len(active)
    scale = cap / m
    x0 = power[active] / scale
    h = problem.interference[:, active] * scale / problem.noise[:, None]
    a = problem.signal * scale / problem.noise
    owner = problem.slot_owner
    q0 = 1.0 + h @ x0
    x = cp.Variable(m, nonneg=True)
    u = cp.Variable(n)
    q = 1.0 + h @ x
    total = q + cp.multiply(a, x[np.searchsorted(active, owner)])
    lower_slot = _BW_OVER_LN2 * (cp.log(total) - np.log(q0) -
                                  cp.multiply(1.0 / q0, q - q0))
    # Explicit incidence avoids a Python loop of scalar CVXPY expressions.
    incidence = np.zeros((n, len(owner)))
    incidence[owner, np.arange(len(owner))] = 1.0
    lower_ratio = cp.multiply(1.0 / problem.demand_bps,
                              incidence @ lower_slot)
    trust = trust_w / scale
    constraints = [x <= x0 + trust, x >= np.maximum(0.0, x0 - trust),
                   cp.sum(x) <= cap / scale, u >= 0, u <= 1,
                   u <= lower_ratio, lower_ratio >= floor_ratio]
    if phase == "sgm":
        constraints.append(cp.sum(u) / n >= anchor_si - 1e-8)
        objective = cp.Maximize(ev.sgm_jacobian[active] @ ((x - x0) * scale))
    else:
        objective = cp.Maximize(cp.sum(u) / n)
    task = cp.Problem(objective, constraints)
    try:
        # CVXPY's solver time limit excludes canonicalization.  Compile first,
        # then give Clarabel only the time still left for this entire call.
        task.get_problem_data(cp.CLARABEL)
        solver_seconds = deadline - perf_counter()
        if solver_seconds <= 0:
            return None, "time_limit_during_canonicalization"
        task.solve(solver="CLARABEL", max_iter=150,
                   time_limit=solver_seconds, verbose=False)
    except cp.SolverError as exc:
        return None, str(exc)
    if task.status not in {"optimal", "optimal_inaccurate"} or x.value is None:
        return None, str(task.status)
    candidate = np.zeros(n)
    candidate[active] = np.maximum(np.asarray(x.value).reshape(-1), 0.0) * scale
    return candidate, (f"{task.status}; compilation_seconds="
                       f"{task.compilation_time}; solver_seconds="
                       f"{task.solver_stats.solve_time}")


def optimize_priority(problem: FixedAllocationPhysics, reference_power,
                      method="sca_si_lp", options=None, callback=None,
                      stop_requested=None):
    """Improve SI first, then SGM under a nondecreasing best-SI anchor.

    The starting A allocation counts as the first physical evaluation.  A
    rejected proposal also consumes a full-scene evaluation.  The exact A
    vector is retained for every solver or validity failure.
    """
    if not isinstance(problem, FixedAllocationPhysics):
        raise TypeError("problem must be FixedAllocationPhysics")
    if method not in {"sca_si_lp", "sca_si_lb"}:
        raise ValueError(f"Unknown priority method: {method}")
    opts = dict(options or {})
    allowed = {"power_cap_w", "maxiter", "max_evaluations", "maxeval",
               "time_limit_seconds", "max_time_seconds", "si_stall_window",
               "si_stall_tolerance", "si_min_evaluations", "si_max_fraction",
               "constraint_tolerance", "safety_margin_ratio",
               "service_margin_ratio", "si_tolerance", "sgm_tolerance",
               "protect_basic_service", "baseline_service_floor_bps"}
    if set(opts) - allowed:
        raise ValueError(f"Unknown optimization options: {sorted(set(opts) - allowed)}")
    cap = float(opts.get("power_cap_w", problem.power_budget_w))
    maxiter = int(opts.get("maxiter", 300))
    maxeval = int(opts.get("max_evaluations", opts.get("maxeval", 2000)))
    limit = float(opts.get("time_limit_seconds", opts.get("max_time_seconds", 30.0)))
    stall_window = int(opts.get("si_stall_window", 200))
    stall_tolerance = float(opts.get("si_stall_tolerance", 1e-6))
    min_evals = int(opts.get("si_min_evaluations", 200))
    max_fraction = float(opts.get("si_max_fraction", 0.85))
    tolerance = float(opts.get("constraint_tolerance", 1e-8))
    margin = float(opts.get("service_margin_ratio",
                            opts.get("safety_margin_ratio", 5e-9)))
    if (opts.get("protect_basic_service", True) is not True or
            not np.isfinite([cap, limit, stall_tolerance, max_fraction,
                             tolerance, margin]).all() or
            not 0 < cap <= min(3000.0, problem.power_budget_w) or
            maxiter < 1 or maxeval < 1 or limit <= 0 or
            stall_window < 1 or min_evals < 1 or not 0 < max_fraction < 1 or
            stall_tolerance < 0 or tolerance != 1e-8 or
            float(opts.get("si_tolerance", 1e-8)) != 1e-8 or
            float(opts.get("sgm_tolerance", 1e-8)) != 1e-8 or
            not 0 <= margin <= tolerance):
        raise ValueError("Invalid priority optimization limits")
    initial = np.asarray(reference_power, dtype=np.float64)
    if initial.shape != (problem.beam_count,) or initial.sum() > cap + 1e-7:
        raise ValueError("Reference power exceeds selected budget or has wrong shape")
    start = perf_counter()
    original = problem.evaluate(initial, with_jacobian=True)
    reference_ratio = original.capacity_bps / problem.demand_bps
    baseline_floor = opts.get("baseline_service_floor_bps")
    if baseline_floor is None:
        baseline_ratio = np.zeros(problem.beam_count)
    else:
        baseline_floor = np.asarray(baseline_floor, dtype=np.float64)
        if (baseline_floor.shape != (problem.beam_count,) or
                not np.isfinite(baseline_floor).all() or
                np.any(baseline_floor < 0) or
                np.any(baseline_floor > 0.1 * problem.demand_bps + 1e-5)):
            raise ValueError("Invalid baseline service floor")
        baseline_ratio = baseline_floor / problem.demand_bps
    # A beam already meeting 0.1D must never newly cross below it.  An
    # initially short beam retains the old 1e-8 ratio numerical tolerance.
    was_met = reference_ratio >= 0.1
    floor_ratio = np.where(was_met, 0.1,
                           np.maximum(reference_ratio, baseline_ratio) - tolerance)
    surrogate_floor = np.where(was_met,
                               np.minimum(reference_ratio, 0.1 + margin),
                               np.minimum(reference_ratio, floor_ratio + margin))
    best_power, best_eval = initial.copy(), original
    anchor_si = original.si
    highest_si_power, highest_si_eval = initial.copy(), original
    evaluations, proposals, subproblems = 1, 0, 0
    surrogate_seconds = 0.0
    history = [{"phase": "initial", "evaluation": 1,
                "final_si_mean": original.si, "final_sgm_mean": original.sgm,
                "power_w": float(initial.sum()), "elapsed_seconds": 0.0}]
    solver_status = []
    termination = "completed"
    phase2_floor = None
    phase1_best_si = original.si

    def global_stop():
        if stop_requested is not None and stop_requested():
            return "stop_requested"
        if perf_counter() - start >= limit:
            return "time_limit"
        if evaluations >= maxeval:
            return "evaluation_limit"
        return None

    def valid(power, ev, phase):
        if (not np.isfinite(power).all() or np.any(power < 0) or
                np.any(power[~problem.active] != 0) or
                power.sum() > cap + 1e-7):
            return False
        ratio = ev.capacity_bps / problem.demand_bps
        if np.any(ratio[was_met] < 0.1):
            return False
        if np.any(ratio[~was_met] < floor_ratio[~was_met]):
            return False
        return phase != "sgm" or ev.si >= anchor_si - 1e-8

    for phase in ("si", "sgm"):
        phase_start = perf_counter()
        phase_evals_start = evaluations
        phase_iterations = accepted = 0
        phase_reason = None
        message = "no proposal"
        trust = max(cap / max(1, np.count_nonzero(problem.active)), 1.0) * 0.5
        if phase == "sgm":
            phase1_best_si = anchor_si
            phase2_floor = anchor_si - 1e-8
        for _ in range(maxiter):
            reason = global_stop()
            if reason:
                termination = phase_reason = reason
                break
            if phase == "si":
                used = evaluations - 1
                elapsed = perf_counter() - start
                if (used >= max_fraction * (maxeval - 1) or
                        elapsed >= max_fraction * limit):
                    phase_reason = "phase_reserve"
                    break
                if used >= max(min_evals, stall_window):
                    # Accepted SI values alone establish progress; the window
                    # spans all true calls, including rejected candidates.
                    old = max((row["final_si_mean"] for row in history
                               if row["phase"] in {"initial", "si"} and
                               row["evaluation"] <= evaluations - stall_window),
                              default=original.si)
                    if anchor_si - old <= stall_tolerance:
                        phase_reason = "si_stall"
                        break
            if not np.any(problem.active):
                phase_reason = "no_active_beams"
                break
            phase_iterations += 1
            proposal_fn = _proposal_lp if method == "sca_si_lp" else _proposal_lb
            begun = perf_counter()
            proposal, message = proposal_fn(
                problem, best_power, best_eval, cap, phase, anchor_si,
                surrogate_floor, trust, limit - (perf_counter() - start))
            subproblems += 1
            surrogate_seconds += perf_counter() - begun
            proposals += 1
            if proposal is None:
                trust *= 0.5
                if trust < 1e-5:
                    phase_reason = "trust_region_small"
                    break
                continue
            direction = proposal - best_power
            if np.max(np.abs(direction)) < 1e-9:
                phase_reason = "surrogate_stationary_direction"
                break
            improved = False
            for backtrack in range(9):
                reason = global_stop()
                if reason:
                    termination = phase_reason = reason
                    break
                step = 2.0 ** -backtrack
                candidate = np.maximum(best_power + step * direction, 0.0)
                candidate[~problem.active] = 0.0
                ev = problem.evaluate(candidate, with_jacobian=True)
                evaluations += 1
                if not valid(candidate, ev, phase):
                    continue
                # The best feasible SI anchor is monotone.  In phase two a
                # higher-SI candidate may leave the old selected point inside
                # the tightened floor; preserve whichever has higher SGM.
                anchor_changed = ev.si > anchor_si + _SI_TIE
                selected_changed = False
                if anchor_changed:
                    anchor_si = ev.si
                    highest_si_power, highest_si_eval = candidate.copy(), ev
                    phase2_floor = anchor_si - 1e-8 if phase == "sgm" else None
                    if (phase == "si" or best_eval.si < phase2_floor or
                            ev.sgm > best_eval.sgm + _SGM_TIE):
                        best_power, best_eval = candidate, ev
                        selected_changed = True
                    improved = True
                elif phase == "si":
                    improved = (ev.si > best_eval.si + _SI_TIE or
                                (abs(ev.si - best_eval.si) <= _SI_TIE and
                                 ev.sgm > best_eval.sgm + _SGM_TIE))
                    if improved:
                        best_power, best_eval = candidate, ev
                        selected_changed = True
                else:
                    improved = (ev.sgm > best_eval.sgm + _SGM_TIE and
                                ev.si >= anchor_si - 1e-8)
                    if improved:
                        best_power, best_eval = candidate, ev
                        selected_changed = True
                if improved:
                    accepted += 1
                    entry = {"phase": phase, "evaluation": evaluations,
                             "final_si_mean": best_eval.si,
                             "final_sgm_mean": best_eval.sgm,
                             "best_si_mean": anchor_si,
                             "power_w": float(best_power.sum()),
                             "anchor_changed": anchor_changed,
                             "selected_changed": selected_changed,
                             "elapsed_seconds": perf_counter() - start,
                             "step_fraction": step}
                    history.append(entry)
                    if callback is not None:
                        callback(dict(entry))
                    break
            if termination != "completed":
                break
            trust = min(cap, trust * 1.4) if improved else trust * 0.5
            if trust < 1e-5:
                phase_reason = "trust_region_small"
                break
        if phase_reason is None:
            phase_reason = "iteration_limit"
        solver_status.append({"phase": phase, "success": accepted > 0,
                              "success_scope": "accepted true feasible improvement only",
                              "stop_reason": phase_reason, "message": message,
                              "iterations": phase_iterations,
                              "evaluations": evaluations - phase_evals_start,
                              "accepted": accepted,
                              "elapsed_seconds": perf_counter() - phase_start})
        if phase == "si":
            phase1_best_si = anchor_si
        if termination != "completed":
            break
        if phase == "sgm":
            termination = phase_reason
    if termination == "completed" and solver_status:
        termination = solver_status[-1]["stop_reason"]
    final_ratio = best_eval.capacity_bps / problem.demand_bps
    error = {
        "budget_excess_w": max(0.0, float(best_power.sum() - cap)),
        "negative_power_w": max(0.0, float(-best_power.min())),
        "empty_slot_power_w": (float(np.max(np.abs(best_power[~problem.active])))
                               if np.any(~problem.active) else 0.0),
        "sgm_shortfall": max(0.0, original.sgm - best_eval.sgm),
        "basic_shortfall_ratio": float(np.maximum(floor_ratio - final_ratio, 0).max()),
        "si_shortfall": max(0.0, original.si - best_eval.si),
        "phase2_si_shortfall": (max(0.0, phase2_floor - best_eval.si)
                                if phase2_floor is not None else 0.0),
    }
    return {
        "method": method, "power_w": best_power.tolist(),
        "initial_metrics": _metrics(problem, initial, original),
        "metrics": _metrics(problem, best_power, best_eval),
        "history": history, "evaluations": evaluations,
        "full_scene_evaluations": evaluations, "scalar_capacity_evaluations": 0,
        "elapsed_seconds": perf_counter() - start,
        "constraint_error": error, "termination_reason": termination,
        "converged": False,
        "convergence_scope": "original SI/SGM stationary point not certified",
        "fallback_used": bool(np.array_equal(best_power, initial)),
        "solver_status": solver_status,
        "solver_power_scale_w": (cap / np.count_nonzero(problem.active)
                                 if np.any(problem.active) else None),
        "warm_start_status": None,
        "algorithm_components": (["si_priority_lp", "true_metric_backtracking"]
                                 if method == "sca_si_lp" else
                                 ["log_capacity_concave_lower_bound",
                                  "si_priority", "true_metric_backtracking"]),
        "surrogate_proposals": proposals,
        "surrogate_subproblems": subproblems,
        "surrogate_elapsed_seconds": surrogate_seconds,
        "surrogate_timing_scope": "CVXPY canonicalization plus solver or LP solver",
        "lp_subproblems": subproblems if method == "sca_si_lp" else 0,
        "lp_elapsed_seconds": surrogate_seconds if method == "sca_si_lp" else 0.0,
        "power_cap_w": cap, "phase1_best_si_mean": phase1_best_si,
        "best_feasible_si_mean": anchor_si, "phase2_si_floor": phase2_floor,
        "highest_si_power_w": highest_si_power.tolist(),
        "highest_si_metrics": _metrics(problem, highest_si_power,
                                        highest_si_eval),
        "sgm_change_from_reference": best_eval.sgm - original.sgm,
        "si_stall_window": stall_window,
        "si_stall_tolerance": stall_tolerance,
        "si_min_evaluations": min_evals,
        "si_max_fraction": max_fraction,
        "safety_margin_ratio": margin,
    }


src/satellite_rl/power_refinement/priority.py 行数 418 SHA256 978d354a2df38b1a6eec9d0525faadb1ee8cb36c6fdd4d09dae22dc2429394d4


## 5. 工作台 runner：初解、A/B/C、逐束核验与保存

读 `evaluate_refinement` 内部的初解身份/SHA 检查、初解容量复算，再读 `verify_solution` 和 `run_refinement_job`。T0 推理先保存初始解；runner 以场景 SHA、重复次数和动作种子核身份，并用 `FixedAllocationPhysics.evaluate` 复算初始容量，然后固定布局和频率槽按 A→B/C 顺序求解。B/C 先读取并校验**已落盘的同一 A 案例**，再分别传入 A 功率。终态物理复算、基本服务与 SI 门槛、引用 SHA、完整时间和逐束数据一并写入每案例 JSON；断点恢复会再次核 SHA 与数值。这里的真实物理验收指按原环境公式重算，不是实测通信数据。`run_refinement_job` 有 CUDA、冻结权重、求解依赖预核，但本笔记本不调用它。

以下代码格为冻结 `src/satellite_rl/power_refinement/runner.py` 的全部源码，保留原函数体、注释和行序。

In [8]:
%%project_module satellite_rl.power_refinement.runner
"""Durable third-stage evaluation using frozen model, scenes and allocations."""
from __future__ import annotations

import csv
import hashlib
import json
import os
from pathlib import Path
import time
import traceback

import numpy as np

from satellite_rl.workbench.runner import (
    _atomic_json, _evaluation_episode, _jsonl, _load_scene, _read_json,
    _sha256, isolated_rng, now, strict_cuda_preflight,
)
from satellite_rl.workbench.storage import clean_json


def _fingerprint(value):
    payload = json.dumps(clean_json(value), sort_keys=True, ensure_ascii=False,
                         separators=(",", ":"), allow_nan=False).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def _optimizer_seed(base, scene_index, repeat, method):
    """Disjoint scene/repeat/method streams for the supported repeat range."""
    if (type(base) is not int or not 0 <= base < 2**32
            or type(scene_index) is not int or scene_index < 0
            or type(repeat) is not int or not 0 <= repeat < 30
            or method not in ("de", "pso")):
        raise ValueError("Invalid optimizer stream identity")
    return (base + 1000 * scene_index + 10 * repeat + {"de": 1, "pso": 2}[method]) % 2**32


def _status(directory, job, **fields):
    previous = _read_json(directory / "status.json", {})
    previous.update(run_id=job["run_id"], heartbeat_at=now(), pid=os.getpid(),
                    research_stage="power_refinement", training_interactions=0,
                    update_step=0, progress_unit="allocation_method_cases", **fields)
    _atomic_json(directory / "status.json", previous)


def _beam_rows(initial, evaluation, power):
    capacities = np.asarray(evaluation.capacity_bps, dtype=float)
    demands = np.asarray([b["required_bps"] for b in initial["beams"]], dtype=float)
    si = np.minimum(capacities / demands, 1.0)
    from satellite_rl.workbench.environment import _sgm, beam_service_fields
    sgm = _sgm(capacities, demands, float(demands.mean()))
    rows = []
    for index, original in enumerate(initial["beams"]):
        rows.append({**original,
                     **beam_service_fields(capacities[index], demands[index], sgm[index]),
                     "power_w": float(power[index]), "si": float(si[index]),
                     "basic_met": bool(capacities[index] >= 0.1 * demands[index]),
                     "initial_capacity_bps": float(original["rate_bps"]),
                     "initial_power_w": float(original["power_w"]),
                     "service_timing": "after_power_refinement_all_interference"})
    return rows


def _priority_solver_contract(solved, reference, options):
    """Check the saved SI anchor and budget independently of the optimizer."""
    phase1 = solved.get("phase1_best_si_mean")
    best = solved.get("best_feasible_si_mean")
    floor = solved.get("phase2_si_floor")
    if phase1 is None or best is None or not np.isfinite([phase1, best]).all():
        raise ValueError("SI优先求解器缺少有效的阶段最佳SI")
    if phase1 < reference["metrics"]["final_si_mean"] - options["si_tolerance"]:
        raise ValueError("SI阶段最佳值低于公共SCA参照")
    if best < phase1 - 1e-12:
        raise ValueError("SI最佳值随阶段推进下降")
    if floor is not None and (not np.isfinite(floor) or
                              abs(floor - (best - options["si_tolerance"])) > 1e-12):
        raise ValueError("SGM阶段锁定SI门槛与最佳可行SI不一致")
    if (solved["evaluations"] > options["maxeval"] or
            reference["solver"]["evaluations"] + solved["evaluations"] > 2 * options["maxeval"]):
        raise ValueError("SI优先完整路线超过冻结容量调用预算")
    return phase1, floor


def verify_solution(initial, beams, power_cap, options, *, reference=None,
                    phase1_best_si=None, locked_si_floor=None):
    """Independent metric acceptance, including zero power and all target beams."""
    initial_beams = initial["beams"]
    if len(beams) != len(initial_beams):
        raise ValueError("优化结果缺失目标波束")
    powers = np.array([b["power_w"] for b in beams])
    capacities = np.array([b["rate_bps"] for b in beams])
    demands = np.array([b["required_bps"] for b in beams])
    if (not np.isfinite(powers).all() or not np.isfinite(capacities).all()
            or np.any(powers < -1e-8) or np.any(capacities < -1e-6)):
        raise ValueError("功率或容量无效")
    if powers.sum() > power_cap + 1e-6:
        raise ValueError("优化结果超过总功率预算")
    for old, new in zip(initial_beams, beams):
        for key in ("beam_id", "original_index", "group", "freq", "slots", "required_bps"):
            if old[key] != new[key]:
                raise ValueError("功率细调改变了冻结波束、需求或频率槽")
        if not new["slots"] and (abs(new["power_w"]) > 1e-8 or abs(new["rate_bps"]) > 1e-6):
            raise ValueError("无槽波束不能产生辐射功率或容量")
    old_c = np.array([b["rate_bps"] for b in initial_beams])
    service_floor = np.minimum(old_c, 0.1 * demands)
    if np.any(capacities < service_floor - np.maximum(1e-3, demands * 1e-8)):
        raise ValueError("优化结果损害原基本服务保护")
    si = float(np.minimum(capacities / demands, 1).mean())
    from satellite_rl.workbench.environment import _sgm
    sgm = float(_sgm(capacities, demands, float(demands.mean())).mean())
    initial_allowance = options["si_tolerance"] + (1e-10 if reference is None else 0.0)
    if si < float(initial["metrics"]["final_si_mean"]) - initial_allowance:
        raise ValueError("最终SI低于初始分配")
    if reference is None and sgm < float(initial["metrics"]["final_sgm_mean"]) - options["sgm_tolerance"] - 1e-10:
        raise ValueError("最终SGM低于初始分配")
    if reference is not None:
        reference_beams = reference["beams"]
        if len(reference_beams) != len(beams):
            raise ValueError("公共SCA参照缺失目标波束")
        reference_c = np.asarray([b["rate_bps"] for b in reference_beams], dtype=float)
        reference_si = float(reference["metrics"]["final_si_mean"])
        if si < reference_si - options["si_tolerance"]:
            raise ValueError("SI优先结果低于公共SCA参照")
        if phase1_best_si is not None and si < float(phase1_best_si) - options["si_tolerance"]:
            raise ValueError("最终SI低于第一阶段最佳可行SI")
        if locked_si_floor is not None and si < float(locked_si_floor):
            raise ValueError("最终SI低于SGM阶段最终锁定门槛")
        basic_threshold = 0.1 * demands
        # A beam already below the threshold must retain its reference capacity;
        # a beam above it must not become a new strict basic-service failure.
        reference_floor = np.where(reference_c >= basic_threshold,
                                   basic_threshold, reference_c)
        if np.any((reference_c >= basic_threshold) & (capacities < basic_threshold)):
            raise ValueError("SI优先结果新增严格基本服务跨下波束")
        if np.any(capacities < reference_floor - np.maximum(1e-3, demands * 1e-8)):
            raise ValueError("SI优先结果新增基本服务跨下或损害原不足波束")
    return {"final_si_mean": si, "final_sgm_mean": sgm,
            "basic_met_count": int(np.sum(capacities >= 0.1 * demands)),
            "all_basic_met": bool(np.all(capacities >= 0.1 * demands)),
            "power_w": float(powers.sum()), "power_budget_w": float(power_cap),
            "throughput_bps": float(capacities.sum()),
            "effective_carried_bps": float(np.minimum(capacities, demands).sum()),
            "zero_capacity_count": int(np.sum(capacities <= 0)),
            "allocated_slots": sum(int(b["slots"]) for b in beams),
            "overload": False, "complete": bool(initial["metrics"]["complete"])}


def _write_csv(path, rows):
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8-sig", newline="") as stream:
        fields = list(dict.fromkeys(key for row in rows for key in row))
        if fields:
            writer = csv.DictWriter(stream, fieldnames=fields)
            writer.writeheader()
            writer.writerows(rows)
    from satellite_rl.training.run_monitor import replace_with_retry
    replace_with_retry(temporary, path)


def _summaries(rows):
    output = {}
    for method in dict.fromkeys(row["method"] for row in rows):
        selected = [row for row in rows if row["method"] == method]
        # Average repeated actions within a scene first; every scene has equal weight.
        scenes = sorted({r["source_scene_sha256"] for r in selected})
        summary = {"scene_count": len(scenes), "allocation_count": len(selected)}
        for key in ("final_si_mean", "final_sgm_mean", "basic_met_count", "all_basic_met",
                    "power_w", "effective_carried_bps", "throughput_bps"):
            summary[key] = float(np.mean([np.mean([r["metrics"][key] for r in selected
                                                  if r["source_scene_sha256"] == scene])
                                         for scene in scenes]))
        durations = [r["solver"]["elapsed_seconds"] for r in selected]
        summary.update(mean_solver_seconds=float(np.mean(durations)),
                       p95_solver_seconds=float(np.percentile(durations, 95)),
                       fallback_count=sum(bool(r["solver"].get("fallback_used")) for r in selected),
                       nonconverged_count=sum(not r["solver"].get("converged", False)
                                              for r in selected),
                       convergence_scope=("not_certified_for_original_objective" if method in ("sca", "sca_si_lp", "sca_si_lb", "wmmse")
                                          else "method_specific_stopping_rule"))
        from satellite_rl.power_refinement.timing import aggregate_scene_timing
        summary.update(aggregate_scene_timing(selected))
        output[method] = summary
    return output


def finalize_results(directory, job, config, rows):
    summaries = _summaries(rows)
    table, beams = [], []
    for row in rows:
        common = {key: row[key] for key in ("run_id", "git_commit", "source_model_sha256",
                                           "scene_id", "source_scene_sha256", "repeat", "method",
                                           "initial_allocation_sha256", "power_budget_mode")}
        if "reference_allocation_sha256" in row:
            common["reference_allocation_sha256"] = row["reference_allocation_sha256"]
        if "reference_case_sha256" in row:
            common["reference_case_sha256"] = row["reference_case_sha256"]
        common["optimizer_seed"] = row.get("optimizer_seed")
        table.append({**common, **row["metrics"],
                      **row.get("timing", {}),
                      "solver_seconds": row["solver"]["elapsed_seconds"],
                      "solver_evaluations": row["solver"]["evaluations"],
                      "reference_solver_evaluations": row.get("reference_solver_evaluations"),
                      "complete_route_solver_evaluations": row.get("complete_route_solver_evaluations"),
                      "phase1_best_si_mean": row["solver"].get("phase1_best_si_mean"),
                      "best_feasible_si_mean": row["solver"].get("best_feasible_si_mean"),
                      "locked_si_floor": row["solver"].get("phase2_si_floor"),
                      "reference_si_mean": (row.get("reference_metrics") or {}).get("final_si_mean"),
                      "reference_sgm_mean": (row.get("reference_metrics") or {}).get("final_sgm_mean"),
                      "complete_route_capacity_budget": row.get("complete_route_capacity_budget"),
                      "complete_route_optimizer_time_budget_seconds": row.get("complete_route_optimizer_time_budget_seconds"),
                      "termination_reason": row["solver"]["termination_reason"],
                      "fallback_used": row["solver"].get("fallback_used", False)})
        beams.extend({**common, **beam} for beam in row["beams"])
    _write_csv(directory / "power_results.csv", table)
    _write_csv(directory / "power_beams.csv", beams)
    scene_table = []
    for method in summaries:
        method_rows = [row for row in rows if row["method"] == method]
        for scene in sorted({row["source_scene_sha256"] for row in method_rows}):
            group = [row for row in method_rows if row["source_scene_sha256"] == scene]
            entry = {"source_scene_sha256": scene, "method": method, "repeats": len(group),
                     "final_si_mean": float(np.mean([r["metrics"]["final_si_mean"] for r in group])),
                     "final_sgm_mean": float(np.mean([r["metrics"]["final_sgm_mean"] for r in group])),
                     "power_w": float(np.mean([r["metrics"]["power_w"] for r in group]))}
            if all("timing" in r for r in group):
                timing_keys = ["initialization_seconds", "physics_setup_seconds", "optimizer_wall_seconds",
                            "optimizer_callback_io_seconds", "optimizer_excluding_callback_seconds",
                            "verification_seconds", "total_pipeline_seconds"]
                if all("reference_total_seconds" in r["timing"] for r in group):
                    timing_keys += ["reference_optimizer_wall_seconds",
                                    "reference_verification_seconds", "reference_total_seconds"]
                for key in timing_keys:
                    entry[key + "_sum"] = sum(r["timing"][key] for r in group)
                    entry[key + "_mean_per_action"] = entry[key + "_sum"] / len(group)
            scene_table.append(entry)
    _write_csv(directory / "power_scene_times.csv", scene_table)
    # Rebuild the canonical complete-case stream; partial attempts remain in iteration logs.
    temporary = directory / "power_cases.jsonl.tmp"
    with temporary.open("w", encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(clean_json(row), ensure_ascii=False, allow_nan=False) + "\n")
    from satellite_rl.training.run_monitor import replace_with_retry
    replace_with_retry(temporary, directory / "power_cases.jsonl")
    # Only committed initial allocations enter the canonical trace stream.
    # Interrupted attempts remain separately identifiable for diagnosis.
    trace_temp = directory / "traces.jsonl.tmp"
    initial_paths = sorted({r["initial_path"] for r in rows})
    with trace_temp.open("w", encoding="utf-8") as stream:
        for path in initial_paths:
            initial = _read_json(path)
            for step in initial["trace"]:
                stream.write(json.dumps({**step, "initial_identity": initial["identity"]},
                                        ensure_ascii=False, allow_nan=False) + "\n")
    replace_with_retry(trace_temp, directory / "traces.jsonl")
    _atomic_json(directory / "results.json", {
        "research_stage": "power_refinement", "summary": summaries,
        "scenes": rows, "baselines": {}, "metadata": {
            "run_id": job["run_id"], "git_commit": job["git_commit"],
            "model_files": job["model_files"], "model_provenance": job.get("model_provenance"),
            "power_refinement": config["power_refinement"], "training_interactions": 0,
            "scope": "Fixed-frequency power refinement; repeated actions are not independent scenes",
            "capacity_semantics": "Environment link capacity; min(C,D) is estimated effective carriage",
        }})
    from satellite_rl.power_refinement.reporting import write_report
    write_report(directory, rows, summaries, config)
    artifacts = {p.name: {"path": str(p), "sha256": _sha256(p)} for p in
                 [directory / name for name in ("results.json", "power_results.csv", "power_beams.csv",
                                                "power_cases.jsonl", "power_scene_times.csv", "power_report.md")]
                 if p.is_file()}
    _atomic_json(directory / "artifacts.json", {"files": artifacts, "created_at": now()})
    return summaries


def evaluate_refinement(directory, job, config, *, device="cuda"):
    """Testable worker body; production entry always performs strict CUDA preflight."""
    from satellite_rl.workbench.checkpoints import load_policy
    from satellite_rl.workbench.datasets import assert_scene_profile, scene_set_files
    from satellite_rl.workbench.environment import WorkbenchEnv
    from satellite_rl.power_refinement.physics import FixedAllocationPhysics
    from satellite_rl.power_refinement.optimizer import optimize_power
    from satellite_rl.power_refinement.config import SI_PRIORITY_METHODS

    directory = Path(directory)
    options = config["power_refinement"]
    for reference in job["model_files"]:
        if _sha256(reference["path"]) != reference["sha256"]:
            raise ValueError("冻结模型SHA不一致")
    policy = load_policy(job["model_path"], config=job["model_config"], device=device)
    dataset = job["scene_sets"]["test"]
    assert_scene_profile(dataset, "europe_v1")
    files = scene_set_files(dataset)
    budget = len(files) * options["repeats"] * len(options["methods"])
    # Scheduler resume flags are operational state, not scientific identity.
    job_identity = {k: job.get(k) for k in
        ("run_id", "git_commit", "source_root", "model_files", "model_config", "scene_sets")}
    if options["methods"] == list(SI_PRIORITY_METHODS):
        job_identity.update(priority_solver_deps_path=job.get("priority_solver_deps_path"),
                            priority_solver_dependency_files=job.get("priority_solver_dependency_files"))
    identity = _fingerprint({"job": job_identity, "config": config})
    manifest_path = directory / "refinement_manifest.json"
    prior = _read_json(manifest_path)
    if prior and prior["identity"] != identity:
        raise ValueError("恢复任务配置或来源与原运行不一致")
    _atomic_json(manifest_path, {"identity": identity, "budget_cases": budget,
                                "scene_files": [(str(p), sha) for p, sha in files]})
    initials_dir = directory / "initial_allocations"
    cases_dir = directory / "power_cases"
    attempts_dir = directory / "initial_attempts"
    initials_dir.mkdir(exist_ok=True)
    cases_dir.mkdir(exist_ok=True)
    attempts_dir.mkdir(exist_ok=True)
    rows = []
    started = time.monotonic()
    stop_requested = lambda: (directory / "stop.request").exists()

    def progress(**fields):
        elapsed = time.monotonic() - started
        if "repeat" in fields:
            fields["repeat_index"] = fields["repeat"]
        _status(directory, job, stage="evaluating", env_step=len(rows), budget_env_steps=budget,
                wall_seconds=elapsed, **fields)

    progress()
    for index, (source, source_sha) in enumerate(files):
        if _sha256(source) != source_sha:
            raise ValueError("冻结场景SHA不一致")
        scene = _load_scene(source)
        for repeat in range(options["repeats"]):
            if stop_requested():
                _status(directory, job, stage="stopped", env_step=len(rows), budget_env_steps=budget)
                return None
            initial_path = initials_dir / f"scene_{index:05d}_r{repeat}.json"
            initial = _read_json(initial_path)
            if initial is None and initial_path.exists():
                raise ValueError("已保存初始分配文件损坏，原文件保留")
            seed = int(options["action_seed"]) + repeat
            initial_identity = _fingerprint([identity, source_sha, index, repeat, seed])
            if initial is None:
                if str(device).startswith("cuda"):
                    import torch
                    torch.cuda.synchronize()
                before = time.monotonic()
                attempt_path = attempts_dir / f"scene_{index:05d}_r{repeat}_{time.time_ns()}.jsonl"
                with isolated_rng(seed):
                    if hasattr(policy, "agent") and hasattr(policy.agent, "rng"):
                        policy.agent.rng.seed(seed + 7321)
                    env = WorkbenchEnv(policy.config)
                    initial = _evaluation_episode(env, policy, scene, f"scene_{index:05d}_r{repeat}",
                                                  seed, options["action_mode"],
                                                  attempt_path, len(rows),
                                                  scene_sha256=source_sha, group="stage3_initial")
                if str(device).startswith("cuda"):
                    torch.cuda.synchronize()
                initial.update(identity=initial_identity, source_scene_sha256=source_sha,
                               inference_seconds=time.monotonic() - before, action_seed=seed,
                               trace_attempt_path=str(attempt_path))
                _atomic_json(initial_path, initial)
            if initial.get("identity") != initial_identity:
                raise ValueError("初始分配与冻结场景或动作种子不一致")
            setup_started = time.perf_counter()
            problem = FixedAllocationPhysics.from_scene(scene, policy.config, initial["beams"])
            p0 = np.array([b["power_w"] for b in initial["beams"]])
            reproduced = problem.evaluate(p0)
            expected = np.array([b["rate_bps"] for b in initial["beams"]])
            if not np.allclose(reproduced.capacity_bps, expected, rtol=1e-8, atol=1e-3):
                raise ValueError("新容量计算未复现原环境全束终态")
            if "physics_setup_seconds" not in initial:
                initial["physics_setup_seconds"] = time.perf_counter() - setup_started
                _atomic_json(initial_path, initial)
            cap = (min(float(p0.sum()), float(options["total_power_w"])) if options["power_budget_mode"] == "initial"
                   else float(options["total_power_w"]))
            allocation_sha = _fingerprint(initial["beams"])
            for method in options["methods"]:
                case_path = cases_dir / f"scene_{index:05d}_r{repeat}_{method}.json"
                reference = None
                reference_sha = None
                reference_case_sha = None
                if method in SI_PRIORITY_METHODS[1:]:
                    reference_path = cases_dir / f"scene_{index:05d}_r{repeat}_sca.json"
                    reference = _read_json(reference_path)
                    if reference is None:
                        raise ValueError("SI优先追加求解必须先完成并保存公共SCA参照")
                    expected_reference_identity = _fingerprint([initial_identity, allocation_sha,
                                                                "sca", options])
                    if reference.get("identity") != expected_reference_identity:
                        raise ValueError("公共SCA参照来源与冻结初始分配不一致")
                    reference_sha = _fingerprint(reference["beams"])
                    reference_case_sha = _sha256(reference_path)
                case_identity = _fingerprint([initial_identity, allocation_sha, method, options]
                                             + ([reference_sha, reference_case_sha] if reference else []))
                row = _read_json(case_path)
                if row is None and case_path.exists():
                    raise ValueError("已保存优化记录损坏，原文件保留")
                if row is not None:
                    if row.get("identity") != case_identity:
                        raise ValueError("已保存优化结果来源不一致")
                    if (method == "sca" and options["methods"] == list(SI_PRIORITY_METHODS)
                            and row.get("reference_allocation_sha256") != _fingerprint(row["beams"])):
                        raise ValueError("已保存公共SCA参照最终分配SHA不一致")
                    if reference is not None and (row.get("reference_allocation_sha256") != reference_sha
                                                  or row.get("reference_case_sha256") != reference_case_sha):
                        raise ValueError("已保存SI优先结果的公共SCA来源不一致")
                    if reference is not None:
                        _priority_solver_contract(row["solver"], reference, options)
                    restored = problem.evaluate([beam["power_w"] for beam in row["beams"]])
                    if not np.allclose(restored.capacity_bps, [b["rate_bps"] for b in row["beams"]],
                                       rtol=1e-8, atol=1e-3):
                        raise ValueError("已保存优化结果容量与物理复算不一致")
                    restored_metrics = verify_solution(initial, row["beams"], cap, options,
                        reference=reference,
                        phase1_best_si=row["solver"].get("phase1_best_si_mean") if reference else None,
                        locked_si_floor=row["solver"].get("phase2_si_floor") if reference else None)
                    for key, value in restored_metrics.items():
                        if key not in row["metrics"] or not np.isclose(row["metrics"][key], value,
                                                                       rtol=1e-10, atol=1e-9):
                            raise ValueError("已保存优化指标与逐束重算不一致")
                    rows.append(row)
                    progress(scene_index=index, repeat=repeat, method=method)
                    continue
                if stop_requested():
                    _status(directory, job, stage="stopped", env_step=len(rows), budget_env_steps=budget)
                    return None
                progress(scene_index=index, repeat=repeat, method=method)

                callback_seconds = 0.0
                def callback(entry):
                    nonlocal callback_seconds
                    callback_started = time.perf_counter()
                    _jsonl(directory / "power_iterations.jsonl", {
                        "time": now(), "scene_id": initial["scene_id"], "repeat": repeat,
                        "method": method, "case_identity": case_identity, **clean_json(entry)})
                    progress(scene_index=index, repeat=repeat, method=method,
                             optimizer_progress=clean_json(entry))
                    callback_seconds += time.perf_counter() - callback_started

                solve_options = {"power_cap_w": cap, "maxiter": options["maxiter"],
                                 "max_evaluations": options["maxeval"],
                                 "time_limit_seconds": options["max_time_seconds"],
                                 "si_tolerance": options["si_tolerance"],
                                 "sgm_tolerance": options["sgm_tolerance"]}
                optimizer_seed = None
                if method in ("de", "pso"):
                    optimizer_seed = _optimizer_seed(options["optimizer_seed"], index, repeat, method)
                    solve_options.update(optimizer_seed=optimizer_seed,
                        **{key: options[key] for key in ("population_size", "de_mutation", "de_crossover",
                                                        "pso_inertia", "pso_cognitive", "pso_social")})
                if reference is not None:
                    from satellite_rl.power_refinement.priority import optimize_priority
                    solve_options.update({key: options[key] for key in
                        ("si_stall_window", "si_stall_tolerance", "si_min_evaluations",
                         "si_max_fraction", "service_margin_ratio")})
                    solve_options["baseline_service_floor_bps"] = np.minimum(
                        np.asarray([beam["rate_bps"] for beam in initial["beams"]], dtype=float),
                        0.1 * problem.demand_bps)
                solver_started = time.perf_counter()
                if reference is None:
                    solved = optimize_power(problem, p0, method=method, options=solve_options,
                                            callback=callback, stop_requested=stop_requested)
                else:
                    solved = optimize_priority(problem,
                        np.asarray([beam["power_w"] for beam in reference["beams"]], dtype=float),
                        method=method, options=solve_options, callback=callback,
                        stop_requested=stop_requested)
                solver_wall = time.perf_counter() - solver_started
                if stop_requested():
                    _status(directory, job, stage="stopped", env_step=len(rows), budget_env_steps=budget)
                    return None
                verify_started = time.perf_counter()
                power = np.asarray(solved["power_w"], dtype=float)
                evaluated = problem.evaluate(power)
                beams = _beam_rows(initial, evaluated, power)
                phase1_best_si = (_priority_solver_contract(solved, reference, options)[0]
                                  if reference is not None else None)
                metrics = verify_solution(initial, beams, cap, options,
                                          reference=reference, phase1_best_si=phase1_best_si,
                                          locked_si_floor=solved.get("phase2_si_floor") if reference else None)
                from satellite_rl.power_refinement.timing import build_timing
                timing = build_timing(initial["inference_seconds"], initial["physics_setup_seconds"],
                                      solver_wall, callback_seconds, time.perf_counter() - verify_started,
                                      **({"reference_optimizer": reference["timing"]["optimizer_wall_seconds"],
                                          "reference_verification": reference["timing"]["verification_seconds"]}
                                         if reference is not None else {}))
                row = {"identity": case_identity, "run_id": job["run_id"],
                       "git_commit": job["git_commit"],
                       "source_model_sha256": next(ref["sha256"] for ref in job["model_files"]
                                                    if Path(ref["path"]).name == "policy.pt"),
                       "scene_id": initial["scene_id"],
                       "source_scene_sha256": source_sha, "source_scene_path": str(source),
                       "repeat": repeat, "method": method, "power_budget_mode": options["power_budget_mode"],
                       "optimizer_seed": optimizer_seed, "timing": timing,
                       "initial_allocation_sha256": allocation_sha, "initial_path": str(initial_path),
                       "initial_metrics": initial["metrics"], "metrics": metrics,
                       "beams": beams, "solver": clean_json(solved),
                       "inference_seconds": initial["inference_seconds"],
                       "service_timing": "after_power_refinement_all_interference"}
                if reference is not None:
                    row.update(reference_allocation_sha256=reference_sha,
                               reference_case_sha256=reference_case_sha,
                               reference_method="sca",
                               reference_case_path=str(reference_path),
                               reference_metrics=reference["metrics"],
                               phase1_best_si_mean=phase1_best_si,
                               best_feasible_si_mean=solved.get("best_feasible_si_mean"),
                               locked_si_floor=solved.get("phase2_si_floor"),
                               reference_solver_evaluations=reference["solver"]["evaluations"],
                               complete_route_solver_evaluations=(
                                   reference["solver"]["evaluations"] + solved["evaluations"]),
                               complete_route_capacity_budget=2 * options["maxeval"],
                               complete_route_optimizer_time_budget_seconds=2 * options["max_time_seconds"],
                               reference_sgm_rule="legacy_sca_protection",
                               priority_sgm_rule="si_first_sgm_may_decline")
                elif method == "sca" and options["methods"] == list(SI_PRIORITY_METHODS):
                    row.update(reference_allocation_sha256=_fingerprint(beams),
                               reference_method="sca", reference_sgm_rule="legacy_sca_protection")
                _atomic_json(case_path, row)
                rows.append(row)
                _jsonl(directory / "evaluations.jsonl", {"env_step": len(rows), "group": method,
                       "scene_id": row["scene_id"], "metrics": metrics})
                _atomic_json(directory / "refinement_checkpoint.json", {
                    "identity": identity, "completed_cases": len(rows),
                    "last_case": str(case_path), "last_case_sha256": _sha256(case_path)})
                progress(scene_index=index, repeat=repeat, method=method, metrics=metrics)
                print(f"{now()} {len(rows)}/{budget} {row['scene_id']} {method} "
                      f"SI={metrics['final_si_mean']:.6f} SGM={metrics['final_sgm_mean']:.6f} "
                      f"P={metrics['power_w']:.3f}W {solved['termination_reason']}", flush=True)
    summary = finalize_results(directory, job, config, rows)
    _status(directory, job, stage="completed", env_step=len(rows), budget_env_steps=budget,
            completed_at=now(), metrics=summary, wall_seconds=time.monotonic() - started)
    return summary


def run_refinement_job(job_dir, *, resume=False):
    directory = Path(job_dir).resolve()
    job = _read_json(directory / "job.json", {})
    try:
        from satellite_rl.power_refinement.config import validate_config
        config = _read_json(directory / "config.json", {})
        if config.get("research_stage") != "power_refinement" or job.get("kind") != "refine":
            raise ValueError("第三阶段任务阶段或类型无效")
        raw_controls = config["power_refinement"]
        config["power_refinement"] = validate_config(raw_controls)
        # Preserve the original identity of jobs frozen before the SI-priority
        # fields existed. Their existing manifest must still be resumable.
        if "si_stall_window" not in raw_controls:
            for key in ("si_stall_window", "si_stall_tolerance", "si_min_evaluations",
                        "si_max_fraction", "service_margin_ratio"):
                config["power_refinement"].pop(key)
        preflight = strict_cuda_preflight(job)
        import scipy
        preflight.update(scipy=scipy.__version__, optimizer_device="cpu", policy_device="cuda")
        from satellite_rl.power_refinement.config import SI_PRIORITY_METHODS
        if config["power_refinement"]["methods"] == list(SI_PRIORITY_METHODS):
            import cvxpy
            import clarabel
            dependency_root = Path(job.get("priority_solver_deps_path", ""))
            if not dependency_root.is_absolute() or not dependency_root.is_dir():
                raise ValueError("冻结的SI优先求解依赖目录无效")
            dependency_root = dependency_root.resolve(strict=True)
            dependency_files = job.get("priority_solver_dependency_files") or []
            expected_files = {str(dependency_root / f"{package}-{version}.dist-info" / name)
                              for package, version in (("cvxpy", "1.6.5"), ("clarabel", "0.11.1"))
                              for name in ("METADATA", "RECORD")}
            if {item.get("path") for item in dependency_files} != expected_files:
                raise ValueError("冻结求解依赖清单不完整")
            for item in dependency_files:
                file = Path(item["path"])
                if not file.is_file() or _sha256(file) != item["sha256"]:
                    raise ValueError("冻结求解依赖清单SHA不一致")
            if (not Path(cvxpy.__file__).resolve().is_relative_to(dependency_root) or
                    not Path(clarabel.__file__).resolve().is_relative_to(dependency_root)):
                raise ValueError("SI优先依赖未从冻结隔离目录导入")
            if cvxpy.__version__ != "1.6.5" or clarabel.__version__ != "0.11.1":
                raise ValueError("SI优先求解依赖版本与冻结配置不一致")
            if "CLARABEL" not in cvxpy.installed_solvers():
                raise ValueError("SI优先求解缺少CLARABEL求解器")
            preflight["priority_solver"] = {
                "dependency_root": str(dependency_root),
                "distribution_manifest_sha256": _fingerprint(dependency_files),
                "cvxpy_version": cvxpy.__version__, "cvxpy_file": cvxpy.__file__,
                "clarabel_version": clarabel.__version__, "clarabel_file": clarabel.__file__,
                "solver": "CLARABEL", "optimizer_device": "cpu"}
        import platform
        preflight.update(cpu_description=platform.processor(), cpu_logical_count=os.cpu_count(),
            cpu_affinity_count=len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None,
            thread_environment={key: os.environ.get(key) for key in
                ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS")})
        _status(directory, job, stage="initializing", preflight=preflight, resume=bool(resume))
        _jsonl(directory / "events.jsonl", {"time": now(), "message": "refinement_worker_started",
                                             "resume": bool(resume), "preflight": preflight})
        return evaluate_refinement(directory, job, config)
    except BaseException as error:
        failure = {"time": now(), "run_id": job.get("run_id"), "error": str(error),
                   "traceback": traceback.format_exc(), "research_stage": "power_refinement"}
        _atomic_json(directory / "failure.json", failure)
        _status(directory, job, stage="failed", error=str(error), traceback=failure["traceback"])
        raise


src/satellite_rl/power_refinement/runner.py 行数 595 SHA256 810978e885760b21bc3a82f7e2632acdc373a5bfc374eec5d15ffbe8ae95167a


## 6. 核对本次真实任务参数与输入链

此格只读取本地运行清单，并核对本次两种预算和冻结来源，**不加载模型也不求解**。S3-P03 的真实输入可在 `outputs/experiments/stage3_si_priority_20261003/remote_archive/{20418,20424}/job/` 查到：`scene_inputs/test/scene_0000000.npz` 至 `scene_0000099.npz` 是 Jan27 共同场景；`model_input/policy.pt` 是 G11 T0 的冻结 150000 步终点（SHA256 `ae095e82…eb824`）；`config.json`、`job.json` 是工作台冻结配置和模型/场景来源，`initial_allocations/` 保存每景三次完整 T0 推理/分配得到的初解，`power_cases/` 保存各 A/B/C 案例。三次推理使用的动作种子从 53001 起。场景集 SHA、原始数据归档 SHA 见运行清单 `scenario` 字段。只有路径和 SHA 同时匹配，才能谈重现；Notebook 中没有凭空合成场景或随机权重。

In [9]:
MANIFEST_PATH = PROJECT_ROOT / "outputs/experiments/stage3_si_priority_20261003/run_manifest.json"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
assert manifest["science_commit"] == SCIENCE_COMMIT
assert manifest["new_training_interactions"] == 0
assert manifest["methods"] == ["sca", "sca_si_lp", "sca_si_lb"]
assert manifest["evaluation_units_total"] == 1800
assert manifest["initializer"]["terminal_env_step"] == 150000
for label in ("total", "initial"):
    config = manifest["groups"][label]["config"]
    assert config["methods"] == manifest["methods"]
    assert (config["maxiter"], config["maxeval"], config["max_time_seconds"]) == (300, 2000, 30.0)
    print(label, "工作台任务", manifest["groups"][label]["run_id"],
          "预算模式", config["power_budget_mode"], "方法", "/".join(config["methods"]))
print("场景", manifest["scenario"]["date_utc"], manifest["scenario"]["scene_count"],
      "权重SHA", manifest["initializer"]["policy_sha256"])


total 工作台任务 refine_sac_20261003_222353_313e994d 预算模式 total 方法 sca/sca_si_lp/sca_si_lb
initial 工作台任务 refine_sac_20261003_222356_db93000c 预算模式 initial 方法 sca/sca_si_lp/sca_si_lb
场景 2024-01-27 100 权重SHA ae095e8246d22b2716ba154315cee18584917d215580821cb53f9aeba78eb824


## 7. S2-P19 的“完整 B”是什么

S2-P19 在另一组第二阶段初解上选择 `none`、`sca`、`sca_si_lp`，因此“完整 B”是**该初解先跑公共 A、再跑 B 追加段**，另保存原分配记录；不等于只调用 B，也不等于重新训练第三阶段模型。[P19 运行清单](../../../outputs/experiments/sac_routes_s2p19_20261007/run_manifest.json)记载的服务提交在 20418/20424/20473 为 `13ea014f1c1709c4a82059fc89ba3092e75010e1`、20474 为 `3cca123c4ed176d4aaa7a66382c29695a1c3bcfb`，I0 终点来源校验另有单文件修复提交 `29dee3248893d42d7569aa57e004cf3db0d0ee65`。不能把 P19 的服务版本直接写成 S3-P03 的 e85a942；两组初解、场景与计时条件也不同。本 Notebook 的代码正文只取 S3-P03 冻结版。

执行路径：`工作台冻结任务 → T0 或 P19 模型产生并保存初解 → 固定频率槽构建 FixedAllocationPhysics → A 优化并保存 → B/C 引用 A 的案例 SHA 分别追加 → 真实物理/约束核验 → 每案例 JSON、汇总、断点`。S3-P03 的 C 是另一条追加路线；P19 选择完整 B 时不包含 C。

## 8. 阅读时易错处与执行边界

- `beam_si` 封顶至 1，`beam_sgm` 是另一指标；报告不能把 SI 上升解释成 SGM 一定上升。
- 物理求值同时考虑所有重叠槽的干扰，最终要核**全体目标束**。候选近似值只用于提议，验收用真实 `problem.evaluate`。
- B/C 的额外 2000 次调用和 30 秒是 **A 之外** 的额度；完整路线预算为最多 4000 次、两段 60 秒。实际可能提前停止，也可能回退。
- 阅读代码或运行 Run All 不产生评价结果。真正调用 `run_refinement_job` 需要工作台生成的冻结任务、T0 权重及场景、依赖目录、DQN/CUDA 和授权的评价预算；请通过工作台任务流程执行。

**本机环境核对（2026-10-12）：**笔记本默认定义加载已在本机 Python/Jupyter 内核通过。本机 `DQN` 解释器的 NumPy 1.26.4、SciPy 1.11.4、PyTorch 2.3.1+cu121 可导入，但未安装 IPython、CVXPY、Clarabel；它无法直接作为这本 Notebook 的 Jupyter 内核运行 B/C 求解。本机也不是 S3-P03 冻结的远端 SciPy 1.14.1/CVXPY 1.6.5/Clarabel 0.11.1 求解环境。这里只核代码定义，不用本机依赖差异重做或解释正式效果。